# Digital Witness — Retail Shoplifting Detection Pipeline

**Student:** Santosh Manoharadas | W1954095 / 20220967  
**Deadline:** 30 March 2026, 1pm  
**Last known results:** MobileNetV2 val_acc=97.7%, BiLSTM val_acc=88.6%

## Pipeline Overview

```
Video Input
    │
    ▼
[CELL 3] YOLO26n Fine-tuning ──► yolo26_dw_v2.pt
    │ (24-class retail detection)
    ▼
[CELL 4] Frame Extraction + MobileNetV2 Training ──► mobilenet_dw.pt
    │ (80/10/10 split, person crops, binary classifier)
    ▼
[CELL 5] BiLSTM + Attention Training ──► bilstm_dw.pt
    │ (sequence of 45 MobileNetV2 features, temporal attention XAI)
    ▼
[CELL 6] Evaluation — MobileNetV2 ──► metrics + confusion matrix
    │
[CELL 7] Evaluation — BiLSTM ──► metrics + attention weight plot (XAI)
    │
    ▼
[CELL 8] PersonProductTracker + POS Integration
    │ (YOLO class IDs → product state → POS cross-check)
    ▼
[CELL 9] Full Inference Pipeline
    │ (run on video: YOLO detect → MobileNetV2 crop → BiLSTM sequence)
    ▼
[CELL 10] Intent Scoring + Bias-Aware Assessment
    │ (weighted formula, quality adjustment, fairness flags)
    ▼
[CELL 11] Case File + Visualisation
    │ (JSON audit trail, behaviour timeline plot, results display)
    ▼
[CELL 12] End-to-End Analysis with POS Verification
```

## Cell 1 — Install Dependencies

In [14]:
# Install all required packages.
# The try/except pattern avoids reinstalling if already present,
# making re-runs faster on the training machine.
import subprocess, sys

def pip_install(package):
    """Install a package via pip if import fails."""
    try:
        __import__(package.split('[')[0].replace('-', '_').split('>=')[0].split('==')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

packages = [
    'opencv-python-headless',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'Pillow',
    'torch',
    'torchvision',
    'ultralytics>=8.3.0',
    'lapx>=0.5.2',
    'reportlab>=4.0.0',
    'tqdm',
    'seaborn',
]

for pkg in packages:
    pip_install(pkg)
    print(f'  OK: {pkg}')

# Verify key libraries after install
import torch
print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
else:
    print('GPU             : N/A (CPU mode)')

  OK: opencv-python-headless
  OK: numpy
  OK: pandas
  OK: matplotlib
  OK: scikit-learn
  OK: Pillow
  OK: torch
  OK: torchvision
  OK: ultralytics>=8.3.0
  OK: lapx>=0.5.2
  OK: reportlab>=4.0.0
  OK: tqdm
  OK: seaborn

PyTorch version : 2.5.1+cu121
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3070 Laptop GPU


## Cell 2 — Configuration

In [15]:
# ── CONFIGURATION — edit this cell to match your machine ──────────────────────────
# All paths, hyperparameters, and constants live here.
# Later cells read from these variables only — no hardcoded values elsewhere.

import platform
import random
from pathlib import Path
import torch
import numpy as np

# ── ENVIRONMENT DETECTION ──────────────────────────────────────────────────────────────
try:
    import google.colab
    ON_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running on Google Colab — Drive mounted.')
except ImportError:
    ON_COLAB = False
    print('Running locally.')

# Auto-detect CUDA — prefer GPU when available for training speed
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Worker count — 0 is required on Windows (cv2 is not fork-safe with multiprocessing)
# NUM_WORKERS=0 on Windows (cv2 not fork-safe) and on CPU-only machines.
# PyTorch's pin_memory background thread raises RuntimeError when workers>0
# and CUDA is unavailable — forcing 0 prevents this on any non-GPU machine.
NUM_WORKERS = 0 if (platform.system() == 'Windows' or device.type != 'cuda') else 4
PIN_MEMORY  = device.type == 'cuda'   # pinned memory only meaningful with CUDA

# ── PATHS ────────────────────────────────────────────────────────────────────────────────
# AUTO-DETECT project root from notebook location.
# In Jupyter, Path().resolve() returns the directory the notebook is in.
# This works on any machine without editing — no hardcoded paths.
ROOT = Path().resolve()

# TRAIN_VIDEOS: folder containing your behaviour video subfolders
# (normal/ and shoplifting/). Edit this to match your local setup.
TRAIN_VIDEOS = ROOT / 'data' / 'videos'   # default — change if videos are elsewhere

# COLAB OVERRIDE
if ON_COLAB:
    ROOT         = Path('/content/drive/MyDrive/DigitalWitness')
    TRAIN_VIDEOS = Path('/content/drive/MyDrive/Dataset')

# Derived paths — all relative to ROOT
MODELS_DIR   = ROOT / 'models'
FRAMES_DIR   = ROOT / 'frames'
OUTPUTS_DIR  = ROOT / 'outputs' / 'cases'
# 4-class behaviour dataset (shoplitingvideo+handpocket v9) — uploaded to project root
DATASET_YAML = ROOT / 'data' / 'dataset' / 'data.yaml'
SEQ_DIR      = ROOT / 'data' / 'sequences'

# Model file paths
YOLO_BASE      = MODELS_DIR / 'yolo26n.pt'
YOLO26_RETAIL  = MODELS_DIR / 'yolo26_dw_v2.pt'
MOBILENET_SAVE = MODELS_DIR / 'mobilenet_dw.pt'
BILSTM_SAVE    = MODELS_DIR / 'bilstm_dw.pt'

# Create output directories if they don't exist
for d in [MODELS_DIR, FRAMES_DIR / 'normal', FRAMES_DIR / 'shoplifting',
          OUTPUTS_DIR, SEQ_DIR / 'normal', SEQ_DIR / 'shoplifting']:
    d.mkdir(parents=True, exist_ok=True)

# ── SMOKE TEST FLAG ────────────────────────────────────────────────────────────────────
# Set SMOKE_TEST = True for quick verification on local CPU.
# Set SMOKE_TEST = False for real training on GPU machine.
SMOKE_TEST = False   # <-- change to False before full training run

EPOCHS_YOLO      = 1  if SMOKE_TEST else 50
EPOCHS_MOBILENET = 1  if SMOKE_TEST else 30
EPOCHS_BILSTM    = 1  if SMOKE_TEST else 20
# BATCH_SIZE — auto-scaled to available GPU VRAM to prevent CUDA OOM.
# Rule of thumb for YOLO at imgsz=640:
#   <4 GB VRAM  → batch 8   (budget / integrated / old GPUs)
#   4–6 GB VRAM → batch 16  (GTX 1060, RTX 3050, laptop GPUs)
#   6–8 GB VRAM → batch 24
#   8+ GB VRAM  → batch 32  (RTX 3070+, Colab A100)
if SMOKE_TEST:
    BATCH_SIZE = 4
elif device.type == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 32 if _vram_gb >= 8 else 24 if _vram_gb >= 6 else 16 if _vram_gb >= 4 else 8
    print(f'GPU VRAM: {_vram_gb:.1f} GB → BATCH_SIZE={BATCH_SIZE}')
else:
    BATCH_SIZE = 8   # CPU fallback

# ── SPLIT RATIOS ──────────────────────────────────────────────────────────────────────────
# 80% train / 10% validation / 10% test — stratified 3-way split
# This is a change from the previous 80/20 train/val — we now have a
# fully held-out test set for final reporting.
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, \
    'Split ratios must sum to 1.0'

# ── MOBILENETV2 HYPERPARAMETERS ─────────────────────────────────────────────────────────────
MOBILENET_INPUT_SIZE  = (224, 224)
MOBILENET_FEATURE_DIM = 1280          # raw MobileNetV2 output before projection
LR_MOBILENET          = 1e-4
FPS_TARGET            = 6             # frames per second to extract from videos
# Why 6fps? A 0.3s concealment gesture = ~2 frames at 6fps.
# Lower fps risks missing brief events. Higher fps bloats storage.

BEHAVIOR_CLASSES = ['normal', 'shoplifting']

# ── BILSTM HYPERPARAMETERS ────────────────────────────────────────────────────────────────────
LSTM_SEQ_LEN    = 45     # 45 frames @ 6fps = 7.5 seconds of context
LSTM_STRIDE     = 15     # stride between windows during inference
LSTM_HIDDEN_DIM = 256    # hidden units per direction (x2 for bidirectional)
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT    = 0.3
LR_LSTM         = 5e-4

# ── INTENT SCORING WEIGHTS ────────────────────────────────────────────────────────────────────
# Based on criminological research (Kim et al., 2021):
# Concealment is the primary physical indicator of intent.
W_BEHAVIOUR    = 0.40   # BiLSTM shoplifting probability
W_CONCEALMENT  = 0.30   # YOLO concealment class detection
W_POS_MISMATCH = 0.20   # items detected vs items billed (POS)
W_DURATION     = 0.10   # proportion of time in suspicious state
assert abs(W_BEHAVIOUR + W_CONCEALMENT + W_POS_MISMATCH + W_DURATION - 1.0) < 1e-9, \
    'Intent scoring weights must sum to 1.0'

# ── THRESHOLDS ────────────────────────────────────────────────────────────────────────────────
THRESHOLD_LOW      = 0.30
THRESHOLD_MEDIUM   = 0.50
THRESHOLD_HIGH     = 0.70
THRESHOLD_CRITICAL = 0.85

# ── YOLO CLASS DEFINITIONS (4-class behaviour dataset) ───────────────────────
# Dataset: shoplitingvideo+handpocket v9 (13,252 images, CC BY 4.0)
# These classes detect behaviour directly — no separate product detector needed.
YOLO_CLASSES = [
    'Looking around',    # 0 — suspicious reconnaissance behaviour
    'Picking-Holding',   # 1 — product interaction (picking up / concealing item)
    'normal',            # 2 — normal shopping behaviour
    'shoplifting',       # 3 — direct shoplifting detection
]

# All 4 classes are person-level detections
PERSON_CLASS_IDS = {0, 1, 2, 3}

# Class 1 (Picking-Holding) indicates product interaction
PRODUCT_HELD_IDS = {1}

# Classes 0 and 3 indicate suspicious/criminal behaviour
CONCEALMENT_IDS = {0, 3}   # Looking around (reconnaissance) + shoplifting

# Direct shoplifting detection class
SHOPLIFTING_CLASS_ID = 3

# Not present in this schema — set to -1 so they never match a detected class
CHECKOUT_OCCUPIED_ID = -1
CHECKOUT_VACANT_ID   = -1

# ── PRINT SUMMARY ────────────────────────────────────────────────────────────────────────────────
print('=' * 50)
print('  DIGITAL WITNESS — CONFIGURATION')
print('=' * 50)
print(f'  Environment  : {"Google Colab" if ON_COLAB else "Local"}')
print(f'  Device       : {device}')
print(f'  GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'  SMOKE TEST   : {SMOKE_TEST}')
print(f'  Epochs YOLO  : {EPOCHS_YOLO}')
print(f'  Epochs MNet  : {EPOCHS_MOBILENET}')
print(f'  Epochs BiLSTM: {EPOCHS_BILSTM}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Split        : 80% train / 10% val / 10% test')
print(f'  ROOT         : {ROOT}')
print(f'  TRAIN_VIDEOS : {TRAIN_VIDEOS} (exists: {TRAIN_VIDEOS.exists()})')
print('=' * 50)

Running locally.
GPU VRAM: 8.6 GB → BATCH_SIZE=32
  DIGITAL WITNESS — CONFIGURATION
  Environment  : Local
  Device       : cuda
  GPU          : NVIDIA GeForce RTX 3070 Laptop GPU
  SMOKE TEST   : False
  Epochs YOLO  : 50
  Epochs MNet  : 30
  Epochs BiLSTM: 20
  Batch size   : 32
  Split        : 80% train / 10% val / 10% test
  ROOT         : D:\Santosh\Project_DigitalWitness
  TRAIN_VIDEOS : D:\Santosh\Project_DigitalWitness\data\videos (exists: True)


## Cell 3 — YOLO26n Fine-tuning

Fine-tunes the YOLO26n base model on the 24-class Roboflow retail annotation dataset. The backbone is frozen (freeze=10) so only the detection head learns retail-specific classes. Output: `models/yolo26_dw_v2.pt`

In [16]:
# ── CELL 3 — YOLO26n Fine-tuning ──────────────────────────────────────────────────────────────────────────
import shutil
import yaml
from ultralytics import YOLO

# Step 1: Verify base weights exist
if not YOLO_BASE.exists():
    raise FileNotFoundError(
        f'Base YOLO weights not found: {YOLO_BASE}\n'
        'Download yolo26n.pt and place it in the models/ directory.'
    )
print(f'Base weights found: {YOLO_BASE}')

# Step 2: Load and verify data.yaml
if not DATASET_YAML.exists():
    raise FileNotFoundError(
        f'Dataset config not found: {DATASET_YAML}\n'
        'Ensure the Roboflow dataset is extracted to data/dataset/.'
    )

with open(DATASET_YAML, 'r') as f:
    yaml_content = yaml.safe_load(f)

num_classes = yaml_content.get('nc', 0)
if num_classes != 4:
    print(f'WARNING: data.yaml has {num_classes} classes but expected 4. '
          f'Check that the shoplitingvideo+handpocket v9 dataset is being used.')
else:
    print(f'data.yaml verified: {num_classes} classes (Looking around, '
          f'Picking-Holding, normal, shoplifting).')

# Step 3: Patch the path field to absolute path for this machine
yaml_content['path'] = str(DATASET_YAML.parent.resolve())
patched_yaml_path = MODELS_DIR / 'data_patched.yaml'
with open(patched_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)
print(f'Patched data.yaml written to: {patched_yaml_path}')

# Step 4: Load the base YOLO model
model = YOLO(str(YOLO_BASE))

# freeze=10 freezes the first 10 layers of the YOLO backbone.
# This preserves low-level feature detectors (edges, textures) learned on COCO,
# while allowing the detection head to learn the 4 behaviour classes.
# The 4-class schema is simpler than the 24-class retail schema,
# so fewer epochs are needed and the head converges faster.
print(f'\nStarting YOLO fine-tuning (4-class behaviour dataset)...')
print(f'  Epochs   : {EPOCHS_YOLO}')
print(f'  Batch    : {BATCH_SIZE}')
print(f'  Device   : {device}')
print(f'  Freeze   : 10 backbone layers')

results = model.train(
    data=str(patched_yaml_path),
    epochs=EPOCHS_YOLO,
    imgsz=416,           # matches Roboflow preprocessing (416×416)
    batch=-1,            # auto-selects safe batch size for available VRAM

    # ── Transfer-learning settings ───────────────────────────────────────────
    # freeze=22 freezes the ENTIRE YOLO26n backbone (all feature-extraction
    # layers) and only trains the detection head.  This is critical:
    # the COCO pre-trained backbone already knows how to detect human shapes,
    # edges, and poses at low level.  Freezing it preserves that knowledge
    # so the fine-tuned model still understands ‘person’ geometry even though
    # the output classes are now behaviour labels, not COCO class names.
    # Unfreezing the backbone (freeze < 10) caused the previous model to
    # forget person-level features and fail to detect people in the app.
    freeze=22,

    # label_smoothing=0.1: distributes 10% of probability mass across
    # all classes during training instead of pushing the target class to 1.0.
    # This prevents the model from becoming overconfident (100% outputs)
    # and improves calibration on ambiguous frames (e.g. someone bending
    # to pick up a dropped item vs genuine concealment).
    label_smoothing=0.1,

    # Lower learning rate for head-only fine-tuning:
    # The head is being trained from near-random initialisation on 4 new
    # classes.  A smaller lr0 prevents large gradient updates that would
    # destabilise the frozen backbone weights adjacent to the head.
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,

    device=str(device),
    patience=15,
    save=True,
    plots=True,
    workers=0,
    project=str(ROOT / 'runs' / 'yolo_ft'),
    name='exp',
    exist_ok=True,
)

# Step 5: Copy best weights to models/
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
if best_weights.exists():
    shutil.copy(best_weights, YOLO26_RETAIL)
    print(f'\nBest weights copied to: {YOLO26_RETAIL}')
else:
    # Fallback: last.pt if best.pt wasn't saved (e.g. 1-epoch smoke test)
    last_weights = Path(results.save_dir) / 'weights' / 'last.pt'
    if last_weights.exists():
        shutil.copy(last_weights, YOLO26_RETAIL)
        print(f'Smoke test: last.pt copied to: {YOLO26_RETAIL}')

# Step 6: Print final metrics
try:
    rd = results.results_dict
    print(f'\nFinal mAP50    : {rd.get("metrics/mAP50(B)", "N/A")}')
    print(f'Final mAP50-95 : {rd.get("metrics/mAP50-95(B)", "N/A")}')
except Exception:
    print('Training complete (metrics not available in smoke-test run).')

print(f'\nYOLO fine-tuning complete. Saved to: {YOLO26_RETAIL}')

Base weights found: D:\Santosh\Project_DigitalWitness\models\yolo26n.pt
data.yaml verified: 4 classes (Looking around, Picking-Holding, normal, shoplifting).
Patched data.yaml written to: D:\Santosh\Project_DigitalWitness\models\data_patched.yaml

Starting YOLO fine-tuning (4-class behaviour dataset)...
  Epochs   : 50
  Batch    : 32
  Device   : cuda
  Freeze   : 10 backbone layers
New https://pypi.org/project/ultralytics/8.4.27 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.21  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Santosh\Project_DigitalWitness\models\data_pat

## Cell 4 — MobileNetV2 Frame Classifier

Extracts frames from behaviour videos at FPS_TARGET (6fps), then trains MobileNetV2 as a binary classifier (normal / shoplifting) on person crops.

**Split:** 80% train / 10% val / 10% test (stratified)  
The test set is held out completely and only used in Cell 6 for final reporting.

In [17]:
# ── CELL 4 — Frame Extraction + MobileNetV2 Training ──────────────────────────────────────────
import cv2
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ── STEP 4a: FRAME EXTRACTION ─────────────────────────────────────────────────────────────────────────────

def extract_frames(video_root, output_dir, classes, fps_target=FPS_TARGET):
    """
    Extract frames from all .mp4 and .avi videos found under
    video_root/{class_name}/ and video_root/*/{class_name}/ (nested layout).

    Saves frames as: output_dir/{class_name}/{video_stem}_frame_{N:05d}.jpg

    Why 6fps? A concealment gesture lasts ~0.3-0.5 seconds.
    At 6fps that's 2-3 frames — enough to capture the event.
    Lower fps risks missing brief suspicious actions entirely.

    Parameters:
        video_root : Path — root directory containing class subdirectories
        output_dir : Path — where extracted frames are saved
        classes    : list[str] — class names to look for
        fps_target : int — target frames per second to extract

    Returns:
        dict mapping class name to number of frames extracted
    """
    counts = {}
    for cls in classes:
        out_cls_dir = output_dir / cls
        out_cls_dir.mkdir(parents=True, exist_ok=True)

        # Support both flat (dataset/normal/*.mp4) and nested (dataset/1/normal/*.mp4)
        videos = (list(video_root.glob(f'{cls}/*.mp4')) +
                  list(video_root.glob(f'{cls}/*.avi')) +
                  list(video_root.glob(f'*/{cls}/*.mp4')) +
                  list(video_root.glob(f'*/{cls}/*.avi')))

        if not videos:
            print(f'  WARNING: No videos found for class "{cls}" under {video_root}')
            counts[cls] = 0
            continue

        frame_count = 0
        for vid_path in tqdm(videos, desc=f'Extracting {cls}'):
            cap = cv2.VideoCapture(str(vid_path))
            if not cap.isOpened():
                print(f'  WARNING: Cannot open video: {vid_path}')
                continue

            # Calculate step size to hit target FPS
            src_fps = cap.get(cv2.CAP_PROP_FPS)
            step = max(1, int(round(src_fps / fps_target)))

            frame_idx = 0
            saved_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                # Save only every `step` frames to achieve target FPS
                if frame_idx % step == 0:
                    out_path = out_cls_dir / f'{vid_path.stem}_frame_{saved_idx:05d}.jpg'
                    cv2.imwrite(str(out_path), frame,
                                [cv2.IMWRITE_JPEG_QUALITY, 85])
                    saved_idx += 1
                    frame_count += 1
                frame_idx += 1
            cap.release()

        counts[cls] = frame_count
        print(f'  {cls}: {frame_count} frames extracted')

    return counts

# Check if frames already exist to avoid re-extraction
normal_frames  = list((FRAMES_DIR / 'normal').glob('*.jpg'))
shop_frames    = list((FRAMES_DIR / 'shoplifting').glob('*.jpg'))

if len(normal_frames) == 0 and len(shop_frames) == 0:
    print('Extracting frames from videos...')
    if TRAIN_VIDEOS.exists():
        counts = extract_frames(TRAIN_VIDEOS, FRAMES_DIR,
                                ['normal', 'shoplifting'])
    else:
        print(f'WARNING: TRAIN_VIDEOS not found at {TRAIN_VIDEOS}')
        print('Skipping frame extraction — ensure videos are present for training.')
        counts = {'normal': 0, 'shoplifting': 0}
else:
    counts = {'normal': len(normal_frames), 'shoplifting': len(shop_frames)}
    print(f'Frames already extracted: normal={counts["normal"]}, '
          f'shoplifting={counts["shoplifting"]}')

# ── STEP 4b: DATASET CLASS ──────────────────────────────────────────────────────────────────────────────

class FrameDataset(Dataset):
    """
    Loads pre-extracted frames on demand (lazy loading).

    Why lazy loading?
    Loading all frames into RAM at once would require:
    ~31,000 frames × 224×224×3 bytes ≈ several GB of RAM.
    Lazy loading reads each image only when the DataLoader requests it,
    keeping memory usage near zero regardless of dataset size.

    Parameters:
        paths     : list[Path] — absolute paths to frame JPEGs
        labels    : list[int]  — corresponding class labels (0=normal, 1=shoplifting)
        transform : torchvision.transforms.Compose — preprocessing pipeline
    """
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        # Read image with OpenCV (BGR), convert to RGB for torchvision
        img = cv2.imread(str(self.paths[idx]))
        if img is None:
            # Return a black frame if file is corrupted — avoids crashing the run
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]

# Define transforms — augmentation only applied to training set
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(MOBILENET_INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(MOBILENET_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── STEP 4c: 3-WAY STRATIFIED SPLIT ────────────────────────────────────────────────────────────────────

# Build file list and labels from the frames directory
all_paths  = []
all_labels = []

for label_idx, cls_name in enumerate(BEHAVIOR_CLASSES):
    cls_frames = sorted((FRAMES_DIR / cls_name).glob('*.jpg'))
    all_paths.extend(cls_frames)
    all_labels.extend([label_idx] * len(cls_frames))

print(f'\nTotal frames available: {len(all_paths)}')
if len(all_paths) == 0:
    print('WARNING: No frames found. Skipping split and model training.')
    print('Run frame extraction first with TRAIN_VIDEOS pointing to your video dataset.')
else:
    # First split: separate test set (10%)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        all_paths, all_labels,
        test_size=TEST_RATIO,
        stratify=all_labels,
        random_state=42,
    )

    # Second split: separate val from trainval (10/90 = 11.1% of trainval)
    val_size_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=val_size_adjusted,
        stratify=y_trainval,
        random_state=42,
    )

    # Save test split to disk to prevent data leakage on partial re-runs
    # Cell 6 loads this file independently — it never touches train/val data
    test_split = {'paths': [str(p) for p in X_test], 'labels': y_test}
    with open(MODELS_DIR / 'mobilenet_test_split.json', 'w') as f:
        json.dump(test_split, f)
    print(f'Test split saved: {len(X_test)} samples → models/mobilenet_test_split.json')

    # Count class distribution per split
    def class_dist(labels):
        c = Counter(labels)
        total = sum(c.values())
        return f'{c[0]/total*100:.0f}% normal / {c[1]/total*100:.0f}% shoplifting'

    print(f'\nSplit sizes:')
    print(f'  Train : {len(X_train)} frames ({class_dist(y_train)})')
    print(f'  Val   : {len(X_val)} frames')
    print(f'  Test  : {len(X_test)} frames (held out — not used until Cell 6)')

    # ── STEP 4d: CLASS IMBALANCE HANDLING ─────────────────────────────────────────────────────────────────
    # Class imbalance: shoplifting videos are rarer than normal videos.
    # Without correction, the model learns to always predict "normal"
    # and achieves misleadingly high accuracy.
    # Fix: WeightedRandomSampler oversamples the minority class (shoplifting)
    # so each batch has a balanced class distribution during training.
    class_counts  = Counter(y_train)
    class_weights = {c: 1.0 / count for c, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in y_train]
    sampler = WeightedRandomSampler(sample_weights, len(y_train), replacement=True)

    # Also add class weights to CrossEntropyLoss for double correction
    loss_weights = torch.tensor(
        [class_weights[0], class_weights[1]], dtype=torch.float32
    ).to(device)
    criterion = nn.CrossEntropyLoss(weight=loss_weights)

    # Create datasets and loaders
    train_ds = FrameDataset(X_train, y_train, transform=train_transform)
    val_ds   = FrameDataset(X_val,   y_val,   transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # ── STEP 4e: MOBILENETV2 MODEL ────────────────────────────────────────────────────────────────────────────

    class MobileNetV2Classifier(nn.Module):
        """
        MobileNetV2 binary classifier for shoplifting behaviour detection.

        Architecture:
            MobileNetV2 features (frozen layers 0-14) → AdaptiveAvgPool2d(1,1)
            → Flatten → Linear(1280, 512) → ReLU → Dropout(0.3)
            → Linear(512, 2)  [normal / shoplifting]

        Why freeze layers 0-14?
        MobileNetV2 has 19 feature layers (0-18). Layers 0-14 detect
        low-level features (edges, gradients) that are universal across
        all image types — freezing them speeds up training significantly.
        Layers 15-18 detect higher-level patterns and are fine-tuned to
        learn retail shoplifting-specific spatial features.

        Why 1280→512 projection?
        MobileNetV2 outputs 1280-dim after global pooling. We project
        to 512-dim to reduce the input size to the BiLSTM in Cell 5,
        keeping sequence memory requirements manageable.

        Why AdaptiveAvgPool2d?
        Without it, Flatten gives 1280*7*7=62720, causing a dimension crash.
        AdaptiveAvgPool2d(1,1) reduces spatial dims to 1×1 regardless of input size.
        """
        def __init__(self, num_classes=2):
            super().__init__()
            # Load pretrained MobileNetV2 using the modern weights API
            base = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

            # Use model.features directly (not children()[:-1])
            self.features = base.features

            # Freeze layers 0-14 to preserve general low-level features
            for i, layer in enumerate(self.features):
                if i <= 14:
                    for param in layer.parameters():
                        param.requires_grad = False

            # Global pooling to collapse spatial dimensions to 1×1
            self.pool = nn.AdaptiveAvgPool2d((1, 1))

            # Classification head with bottleneck projection
            self.classifier = nn.Sequential(
                nn.Linear(MOBILENET_FEATURE_DIM, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.3),
                nn.Linear(512, num_classes),
            )

        def forward(self, x):
            x = self.features(x)       # (B, 1280, H, W)
            x = self.pool(x)           # (B, 1280, 1, 1)
            x = torch.flatten(x, 1)   # (B, 1280)
            x = self.classifier(x)    # (B, num_classes)
            return x

        def extract_features(self, x):
            """Return 1280-dim feature vector (used by BiLSTM in Cell 5)."""
            x = self.features(x)
            x = self.pool(x)
            x = torch.flatten(x, 1)
            return x

    model = MobileNetV2Classifier(num_classes=2).to(device)

    # Optimizer only updates unfrozen parameters
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_MOBILENET,
    )
    # Halve learning rate every 8 epochs
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

    # ── STEP 4f: TRAINING LOOP ────────────────────────────────────────────────────────────────────────────

    best_val_acc    = 0.0
    patience_count  = 0
    EARLY_STOP_PAT  = 5   # stop if val_loss doesn't improve for this many epochs
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print('\nStarting MobileNetV2 training...')
    print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Acc":>7}  {"LR":>8}')
    print('-' * 50)

    for epoch in range(EPOCHS_MOBILENET):
        # ── Training phase ──
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total   = 0

        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS_MOBILENET}',
                                  leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * imgs.size(0)
            preds          = outputs.argmax(1)
            train_correct += (preds == labels).sum().item()
            train_total   += imgs.size(0)

        train_acc  = train_correct / max(train_total, 1)
        avg_loss   = train_loss    / max(train_total, 1)

        # ── Validation phase ──
        model.eval()
        val_correct = 0
        val_total   = 0
        val_loss_sum = 0.0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
                val_loss_sum += loss.item() * imgs.size(0)
                preds         = outputs.argmax(1)
                val_correct  += (preds == labels).sum().item()
                val_total    += imgs.size(0)

        val_acc  = val_correct / max(val_total, 1)
        val_loss = val_loss_sum / max(val_total, 1)
        current_lr = scheduler.get_last_lr()[0]

        print(f'{epoch+1:>5}  {avg_loss:>10.4f}  {train_acc*100:>8.1f}%  '
              f'{val_acc*100:>6.1f}%  {current_lr:>8.6f}')

        history['train_loss'].append(avg_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Save best model checkpoint
        if val_acc > best_val_acc:
            best_val_acc   = val_acc
            patience_count = 0
            torch.save({
                'state_dict':     model.state_dict(),
                'val_acc':        best_val_acc,
                'epochs_trained': epoch + 1,
                'feature_dim':    MOBILENET_FEATURE_DIM,
                'classes':        BEHAVIOR_CLASSES,
            }, MOBILENET_SAVE)
        else:
            patience_count += 1

        scheduler.step()

        # Early stopping — prevents overfitting on small datasets
        if patience_count >= EARLY_STOP_PAT:
            print(f'\nEarly stopping triggered at epoch {epoch+1} '
                  f'(no val improvement for {EARLY_STOP_PAT} epochs).')
            break

    print('\n' + '=' * 55)
    print('  MobileNetV2 Training Complete')
    print(f'  Best val accuracy : {best_val_acc*100:.1f}%')
    print(f'  Saved to          : {MOBILENET_SAVE}')
    print('  NOTE: Final test-set evaluation is in Cell 6.')
    print('  Do not use val accuracy as the reported metric — test accuracy is definitive.')
    print('=' * 55)

    MOBILENET_HISTORY_PATH = MODELS_DIR / 'mobilenet_history.json'
    with open(MOBILENET_HISTORY_PATH, 'w') as hf:
        json.dump(history, hf, indent=2)
    print(f'Training history saved to: {MOBILENET_HISTORY_PATH}')

Frames already extracted: normal=32050, shoplifting=61575

Total frames available: 93625
Test split saved: 9363 samples → models/mobilenet_test_split.json

Split sizes:
  Train : 74899 frames (34% normal / 66% shoplifting)
  Val   : 9363 frames
  Test  : 9363 frames (held out — not used until Cell 6)

Starting MobileNetV2 training...
Epoch  Train Loss  Train Acc  Val Acc        LR
--------------------------------------------------


    1      0.0411      98.0%    99.4%  0.000100


    2      0.0146      99.4%    99.5%  0.000100


    3      0.0121      99.5%    99.6%  0.000100


    4      0.0095      99.6%    99.3%  0.000100


...

   13      0.0054      99.8%    99.7%  0.000050


   14      0.0056      99.8%    99.7%  0.000050


   15      0.0060      99.7%    99.7%  0.000050


   16      0.0060      99.7%    99.7%  0.000050


   17      0.0050      99.8%    99.7%  0.000025

Early stopping triggered at epoch 17 (no val improvement for 5 epochs).

  MobileNetV2 Training Complete
  Best val accuracy : 99.7%
  Saved to          : D:\Santosh\Project_DigitalWitness\models\mobilenet_dw.pt
  NOTE: Final test-set evaluation is in Cell 6.
  Do not use val accuracy as the reported metric — test accuracy is definitive.
Training history saved to: D:\Santosh\Project_DigitalWitness\models\mobilenet_history.json


## Cell 5 — BiLSTM + Temporal Attention Classifier

Trains a Bidirectional LSTM with a learned attention mechanism on sequences of MobileNetV2 feature vectors extracted from behaviour videos.

This is the core temporal reasoning component of Digital Witness. A single frame cannot distinguish intentional concealment from legitimate bag repacking. The BiLSTM analyses 7.5 seconds (45 frames) of context, and the attention mechanism learns which specific frames drove the classification. Those attention weights are the XAI (Explainable AI) output of the system.

**Split:** Same 80/10/10 strategy. Test set held out until Cell 7.

In [18]:
# ── CELL 5 — BiLSTM + Temporal Attention Training ─────────────────────────────────────────────
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ── STEP 5a: FEATURE EXTRACTION FROM VIDEOS ───────────────────────────────────────────────────────────────────

def extract_features_from_video(video_path, mobilenet_model, fps_target=FPS_TARGET):
    """
    Extract a sequence of MobileNetV2 feature vectors from a video.

    For each frame at fps_target:
      1. Read frame with cv2
      2. Apply inference transform (no augmentation)
      3. Pass through MobileNetV2 feature extractor (eval mode, no_grad)
      4. Append 1280-dim vector to sequence

    Returns: np.array of shape (num_frames, 1280)

    Why pre-extract and cache?
    Running MobileNetV2 forward pass on every frame during LSTM training
    would be extremely slow (each batch requires ~45 × batch_size forward passes).
    We extract once and cache to disk, then the LSTM training only loads
    the pre-computed numpy arrays.

    Parameters:
        video_path      : Path — input video file
        mobilenet_model : MobileNetV2Classifier in eval mode on correct device
        fps_target      : int — frames per second to sample
    """
    import cv2
    infer_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(MOBILENET_INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return np.zeros((1, MOBILENET_FEATURE_DIM), dtype=np.float32)

    src_fps = cap.get(cv2.CAP_PROP_FPS)
    step    = max(1, int(round(src_fps / fps_target)))
    features = []

    frame_idx = 0
    mobilenet_model.eval()
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % step == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                tensor    = infer_transform(frame_rgb).unsqueeze(0).to(device)
                feat      = mobilenet_model.extract_features(tensor)
                features.append(feat.cpu().numpy()[0])
            frame_idx += 1
    cap.release()

    if not features:
        return np.zeros((1, MOBILENET_FEATURE_DIM), dtype=np.float32)
    return np.array(features, dtype=np.float32)   # (T, 1280)

# Load trained MobileNetV2 for feature extraction
if MOBILENET_SAVE.exists():
    # Re-define MobileNetV2Classifier here (needed if Cell 4 was not run this session)
    from torchvision import models as tv_models

    class _MNetV2(nn.Module):
        def __init__(self):
            super().__init__()
            base = tv_models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
            self.features    = base.features
            self.pool        = nn.AdaptiveAvgPool2d((1, 1))
            self.classifier  = nn.Sequential(
                nn.Linear(MOBILENET_FEATURE_DIM, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.3),
                nn.Linear(512, 2),
            )
        def forward(self, x):
            x = self.features(x)
            x = self.pool(x)
            x = torch.flatten(x, 1)
            return self.classifier(x)
        def extract_features(self, x):
            x = self.features(x)
            x = self.pool(x)
            return torch.flatten(x, 1)

    mnet = _MNetV2().to(device)
    ckpt = torch.load(MOBILENET_SAVE, map_location=device)
    # Handle both save formats: new {'state_dict':...} dict and legacy raw state dict
    sd = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    mnet.load_state_dict(sd)
    mnet.eval()
    print(f'MobileNetV2 loaded from {MOBILENET_SAVE}')
else:
    print(f'WARNING: {MOBILENET_SAVE} not found. Run Cell 4 first.')
    mnet = None

# Extract and cache sequences as .npy files
if mnet is not None:
    for cls_name in BEHAVIOR_CLASSES:
        out_cls_dir = SEQ_DIR / cls_name
        out_cls_dir.mkdir(parents=True, exist_ok=True)

        # Find all videos for this class
        vids = (list(TRAIN_VIDEOS.glob(f'{cls_name}/*.mp4')) +
                list(TRAIN_VIDEOS.glob(f'{cls_name}/*.avi')) +
                list(TRAIN_VIDEOS.glob(f'*/{cls_name}/*.mp4')) +
                list(TRAIN_VIDEOS.glob(f'*/{cls_name}/*.avi')))

        for vid in tqdm(vids, desc=f'Extracting sequences: {cls_name}'):
            npy_path = out_cls_dir / f'{vid.stem}.npy'
            if npy_path.exists():
                continue   # skip already-extracted sequences
            seq = extract_features_from_video(vid, mnet)
            np.save(str(npy_path), seq)

    print('Sequence feature extraction complete.')

# ── STEP 5b: SEQUENCE DATASET ─────────────────────────────────────────────────────────────────────────────────

class SequenceDataset(Dataset):
    """
    Loads pre-extracted feature sequences and returns fixed-length windows.

    Sliding window: given a sequence of N frames, returns windows of
    length LSTM_SEQ_LEN with stride LSTM_STRIDE. This data augmentation
    creates multiple training examples from each video.

    If a sequence is shorter than LSTM_SEQ_LEN, it is zero-padded on
    the right. This ensures all batches have the same tensor dimensions.

    Parameters:
        seq_paths : list[Path] — paths to .npy sequence files
        labels    : list[int]  — class labels (0=normal, 1=shoplifting)
        seq_len   : int        — fixed window length
        stride    : int        — stride between consecutive windows
    """
    def __init__(self, seq_paths, labels, seq_len=LSTM_SEQ_LEN, stride=LSTM_STRIDE):
        self.seq_len = seq_len
        self.stride  = stride
        # Build index: (file_index, window_start)
        self.index = []
        self.labels_list = []
        for i, (p, lbl) in enumerate(zip(seq_paths, labels)):
            seq = np.load(str(p))
            T   = seq.shape[0]
            # Create sliding windows over the sequence
            starts = list(range(0, max(1, T - seq_len + 1), stride))
            self.index.extend([(p, s) for s in starts])
            self.labels_list.extend([lbl] * len(starts))
        self._cache = {}   # optional in-memory cache for small datasets

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        path, start = self.index[idx]
        label       = self.labels_list[idx]
        seq = np.load(str(path))
        window = seq[start: start + self.seq_len]

        # Zero-pad if sequence is shorter than LSTM_SEQ_LEN
        if window.shape[0] < self.seq_len:
            pad = np.zeros((self.seq_len - window.shape[0], MOBILENET_FEATURE_DIM),
                           dtype=np.float32)
            window = np.concatenate([window, pad], axis=0)

        return torch.tensor(window, dtype=torch.float32), label

# Build sequence file lists
seq_paths_all  = []
seq_labels_all = []

for label_idx, cls_name in enumerate(BEHAVIOR_CLASSES):
    files = sorted((SEQ_DIR / cls_name).glob('*.npy'))
    seq_paths_all.extend(files)
    seq_labels_all.extend([label_idx] * len(files))

print(f'Total sequences: {len(seq_paths_all)}')

if len(seq_paths_all) > 0:
    # 3-way stratified split on sequences (not frames)
    X_sv, X_stest, y_sv, y_stest = train_test_split(
        seq_paths_all, seq_labels_all,
        test_size=TEST_RATIO, stratify=seq_labels_all, random_state=42,
    )
    val_adj = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_strain, X_sval, y_strain, y_sval = train_test_split(
        X_sv, y_sv, test_size=val_adj, stratify=y_sv, random_state=42,
    )

    # Save test split to disk — prevents data leakage on partial re-runs
    bilstm_test_split = {
        'paths': [str(p) for p in X_stest],
        'labels': y_stest,
    }
    with open(MODELS_DIR / 'bilstm_test_split.json', 'w') as f:
        json.dump(bilstm_test_split, f)
    print(f'BiLSTM test split saved: {len(X_stest)} sequences')

    # Build datasets and dataloaders
    train_seq_ds = SequenceDataset(X_strain, y_strain)
    val_seq_ds   = SequenceDataset(X_sval,   y_sval)

    # Oversample minority class in training
    seq_class_counts  = Counter(y_strain)
    seq_class_weights = {c: 1.0 / cnt for c, cnt in seq_class_counts.items()}
    seq_sample_w = [seq_class_weights[lbl] for lbl in train_seq_ds.labels_list]
    seq_sampler  = WeightedRandomSampler(seq_sample_w, len(seq_sample_w), replacement=True)

    seq_loss_w = torch.tensor(
        [seq_class_weights[0], seq_class_weights[1]], dtype=torch.float32
    ).to(device)
    seq_criterion = nn.CrossEntropyLoss(weight=seq_loss_w)

    seq_train_loader = DataLoader(train_seq_ds, batch_size=BATCH_SIZE, sampler=seq_sampler,
                                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    seq_val_loader   = DataLoader(val_seq_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── STEP 5c: TEMPORAL ATTENTION ───────────────────────────────────────────────────────────────────────────────────

class TemporalAttention(nn.Module):
    """
    Learned temporal attention over LSTM hidden states.

    For a sequence of T hidden states h_1, ..., h_T:

        e_t = tanh(W_1 * h_t)    # score each timestep
        a_t = softmax(e_t)        # normalise to probability distribution
        c   = sum(a_t * h_t)      # weighted sum = context vector

    The attention weights a_1, ..., a_T tell us:
    'Which of the 45 frames in this window contributed most to the decision?'

    These weights are the XAI output. A high weight on frame 23 means
    frame 23 was the most suspicious moment in the sequence.
    This is equivalent to LIME's local explanation principle but
    architecturally integrated rather than post-hoc computed.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden_dim)
        scores  = self.attn(lstm_output).squeeze(-1)      # (batch, seq_len)
        weights = torch.softmax(scores, dim=1)             # (batch, seq_len)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

# ── STEP 5d: BILSTM MODEL ────────────────────────────────────────────────────────────────────────────────────────

class BiLSTMAttentionClassifier(nn.Module):
    """
    Bidirectional LSTM with temporal attention for shoplifting intent classification.

    Architecture:
        Input (batch, seq_len=45, feature_dim=1280)
            → BiLSTM (2 layers, hidden=256 per direction)
            → Output (batch, seq_len, 512)  [256 forward + 256 backward]
            → TemporalAttention
            → Context vector (batch, 512)
            → Dropout(0.3)
            → Linear(512, 2)  [normal / shoplifting]

    Why Bidirectional?
    A forward-only LSTM at frame 23 knows what happened in frames 1-22
    but not 24-45. Shoplifting intent often requires post-event context:
    e.g. a concealment gesture (frame 23) followed immediately by walking
    toward the exit (frames 24-30) is more suspicious than the same
    gesture followed by walking to the checkout. Bidirectionality lets
    the model see both directions of context at every timestep.

    Why Attention?
    Without attention, we'd use only the final hidden state as a summary.
    Attention creates a weighted combination of ALL hidden states, allowing
    the model to focus on the most relevant frames rather than relying
    solely on what the LSTM remembers at the end of the sequence.
    """
    def __init__(self, feature_dim=MOBILENET_FEATURE_DIM,
                 hidden_dim=LSTM_HIDDEN_DIM, num_layers=LSTM_NUM_LAYERS,
                 num_classes=2, dropout=LSTM_DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,   # doubled hidden dim: hidden_dim*2
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # hidden_dim*2 because bidirectional
        self.attention  = TemporalAttention(hidden_dim * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, feature_dim)
        lstm_out, _ = self.bilstm(x)          # (batch, seq_len, hidden*2)
        context, attn_weights = self.attention(lstm_out)
        out = self.dropout(context)
        out = self.classifier(out)
        return out, attn_weights

# ── STEP 5e: TRAINING LOOP ────────────────────────────────────────────────────────────────────────────────────────

if len(seq_paths_all) > 0:
    bilstm_model = BiLSTMAttentionClassifier().to(device)
    bilstm_opt   = torch.optim.Adam(bilstm_model.parameters(), lr=LR_LSTM)
    bilstm_sched = torch.optim.lr_scheduler.StepLR(bilstm_opt, step_size=8, gamma=0.5)

    best_val_acc_lstm = 0.0
    patience_lstm     = 0
    EARLY_STOP_PAT    = 5
    history_b = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print('Starting BiLSTM training...')
    print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Acc":>7}')
    print('-' * 45)

    for epoch in range(EPOCHS_BILSTM):
        # ── Training phase ──
        bilstm_model.train()
        t_loss = 0.0
        t_correct = 0
        t_total   = 0

        for seqs, labels in tqdm(seq_train_loader,
                                  desc=f'BiLSTM Epoch {epoch+1}/{EPOCHS_BILSTM}',
                                  leave=False):
            seqs, labels = seqs.to(device), labels.to(device)
            bilstm_opt.zero_grad()
            logits, _ = bilstm_model(seqs)
            loss      = seq_criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=5.0)
            bilstm_opt.step()

            t_loss    += loss.item() * seqs.size(0)
            preds      = logits.argmax(1)
            t_correct += (preds == labels).sum().item()
            t_total   += seqs.size(0)

        t_acc = t_correct / max(t_total, 1)
        avg_l = t_loss    / max(t_total, 1)

        # ── Validation phase ──
        bilstm_model.eval()
        v_correct = 0
        v_total   = 0

        with torch.no_grad():
            for seqs, labels in seq_val_loader:
                seqs, labels = seqs.to(device), labels.to(device)
                logits, _    = bilstm_model(seqs)
                preds         = logits.argmax(1)
                v_correct    += (preds == labels).sum().item()
                v_total      += seqs.size(0)

        v_acc = v_correct / max(v_total, 1)
        print(f'{epoch+1:>5}  {avg_l:>10.4f}  {t_acc*100:>8.1f}%  {v_acc*100:>6.1f}%')

        history_b['train_loss'].append(avg_l)
        history_b['val_loss'].append(0.0)
        history_b['train_acc'].append(t_acc)
        history_b['val_acc'].append(v_acc)

        # Save best model
        if v_acc > best_val_acc_lstm:
            best_val_acc_lstm = v_acc
            patience_lstm     = 0
            torch.save({
                'state_dict': bilstm_model.state_dict(),
                'val_acc':    best_val_acc_lstm,
                'config': {
                    'feature_dim': MOBILENET_FEATURE_DIM,
                    'hidden_dim':  LSTM_HIDDEN_DIM,
                    'num_layers':  LSTM_NUM_LAYERS,
                    'seq_len':     LSTM_SEQ_LEN,
                    'classes':     BEHAVIOR_CLASSES,
                },
            }, BILSTM_SAVE)
        else:
            patience_lstm += 1

        bilstm_sched.step()

        if patience_lstm >= EARLY_STOP_PAT:
            print(f'Early stopping at epoch {epoch+1}.')
            break

    print(f'\nBiLSTM Training Complete. Best val acc: {best_val_acc_lstm*100:.1f}%')
    print(f'Saved to: {BILSTM_SAVE}')

    BILSTM_HISTORY_PATH = MODELS_DIR / 'bilstm_history.json'
    with open(BILSTM_HISTORY_PATH, 'w') as hf:
        json.dump(history_b, hf, indent=2)
    print(f'Training history saved to: {BILSTM_HISTORY_PATH}')
else:
    print('No sequences found — skipping BiLSTM training. Run Cell 5a first.')

C:\Users\johnf\AppData\Local\Temp\ipykernel_5420\4149055312.py:111: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MOBILENET_SAVE, map_location=device)


MobileNetV2 loaded from D:\Santosh\Project_DigitalWitness\models\mobilenet_dw.pt


Extracting sequences: shoplifting: 100%|██████████| 134/134 [00:00<00:00, 11636.13it/s]


Sequence feature extraction complete.
Total sequences: 263
BiLSTM test split saved: 27 sequences
Starting BiLSTM training...
Epoch  Train Loss  Train Acc  Val Acc
---------------------------------------------


    1      0.0236      99.4%   100.0%


    2      0.0083      99.8%   100.0%


    3      0.0059      99.8%    99.4%


    4      0.0040      99.8%   100.0%


    5      0.0059      99.8%    99.4%


    6      0.0029      99.9%   100.0%
Early stopping at epoch 6.

BiLSTM Training Complete. Best val acc: 100.0%
Saved to: D:\Santosh\Project_DigitalWitness\models\bilstm_dw.pt
Training history saved to: D:\Santosh\Project_DigitalWitness\models\bilstm_history.json


## Cell 6 — MobileNetV2 Evaluation on Held-Out Test Set

Loads the saved test split from Cell 4 and evaluates the trained model.  
This is the definitive performance report — the model has never seen these samples during training or validation.

In [19]:
# ── CELL 6 — MobileNetV2 Evaluation (TEST SET) ────────────────────────────────────────────────────────────────
import json
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from torchvision import transforms, models
from torchvision.models import MobileNet_V2_Weights
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ── STEP 1: Load test split ──────────────────────────────────────────────────────────────────────────────────────
test_split_path = MODELS_DIR / 'mobilenet_test_split.json'
if not test_split_path.exists():
    raise FileNotFoundError(
        f'Test split not found: {test_split_path}\n'
        'Run Cell 4 first to generate the test split.'
    )

with open(test_split_path, 'r') as f:
    test_split = json.load(f)

X_test_paths = [Path(p) for p in test_split['paths']]
y_test       = test_split['labels']
print(f'Test split loaded: {len(X_test_paths)} samples')

# ── STEP 2: Load trained model ───────────────────────────────────────────────────────────────────────────────
if not MOBILENET_SAVE.exists():
    raise FileNotFoundError(
        f'Trained model not found: {MOBILENET_SAVE}\n'
        'Run Cell 4 to train MobileNetV2 first.'
    )

# Re-define the model architecture (needed if running Cell 6 in isolation)
class MobileNetV2Eval(nn.Module):
    """MobileNetV2 binary classifier — identical architecture to Cell 4."""
    def __init__(self):
        super().__init__()
        base            = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        self.features   = base.features
        self.pool       = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(MOBILENET_FEATURE_DIM, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, 2),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

eval_model = MobileNetV2Eval().to(device)
ckpt = torch.load(MOBILENET_SAVE, map_location=device)
eval_model.load_state_dict(ckpt['state_dict'])
eval_model.eval()
print(f'Model loaded (trained for {ckpt["epochs_trained"]} epochs, '
      f'best val acc: {ckpt["val_acc"]*100:.1f}%)')

# Reuse val_transform (no augmentation for evaluation)
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(MOBILENET_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class EvalFrameDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = cv2.imread(str(self.paths[idx]))
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img), self.labels[idx]

test_ds     = EvalFrameDataset(X_test_paths, y_test, eval_transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         num_workers=NUM_WORKERS)

# ── STEP 3: Run inference on all test samples ─────────────────────────────────────────────────────────────────
all_preds = []
all_true  = []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Evaluating MobileNetV2'):
        imgs   = imgs.to(device)
        logits = eval_model(imgs)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_true.extend(labels.numpy().tolist())

# ── STEP 4: Compute metrics ────────────────────────────────────────────────────────────────────────────────────────
acc  = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, average='weighted', zero_division=0)
rec  = recall_score(all_true, all_preds, average='weighted', zero_division=0)
f1   = f1_score(all_true, all_preds, average='weighted', zero_division=0)
cm   = confusion_matrix(all_true, all_preds)
cr   = classification_report(all_true, all_preds,
                              target_names=BEHAVIOR_CLASSES, zero_division=0)

print('=' * 60)
print('  MOBILENET V2 — TEST SET EVALUATION')
print('=' * 60)
print(f'  Test samples      : {len(all_true)}')
print(f'  Accuracy          : {acc*100:.2f}%')
print(f'  Precision         : {prec*100:.2f}%')
print(f'  Recall            : {rec*100:.2f}%')
print(f'  F1 Score          : {f1*100:.2f}%')
print('-' * 60)
print('  Per-class:')
for i, cls_name in enumerate(BEHAVIOR_CLASSES):
    row = cr.split('\n')[i + 2].split()
    if len(row) >= 4:
        print(f'    {cls_name:<12} → precision: {row[1]}  recall: {row[2]}  f1: {row[3]}')
print('=' * 60)
print('\nFull classification report:')
print(cr)

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('MobileNetV2 — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()

cm_save_path = OUTPUTS_DIR / 'mobilenet_confusion_matrix.png'
plt.savefig(cm_save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved to: {cm_save_path}')

# ── STEP 5: Save metrics to JSON ──────────────────────────────────────────────────────────────────────────────────
eval_results = {
    'model': 'MobileNetV2',
    'test_samples': len(all_true),
    'accuracy':  float(acc),
    'precision': float(prec),
    'recall':    float(rec),
    'f1_score':  float(f1),
    'confusion_matrix': cm.tolist(),
}
eval_json_path = MODELS_DIR / 'mobilenet_eval.json'
with open(eval_json_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'Metrics saved to: {eval_json_path}')

Test split loaded: 9363 samples


C:\Users\johnf\AppData\Local\Temp\ipykernel_5420\454304683.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MOBILENET_SAVE, map_location=device)


Model loaded (trained for 12 epochs, best val acc: 99.7%)


Evaluating MobileNetV2: 100%|██████████| 293/293 [01:21<00:00,  3.61it/s]


  MOBILENET V2 — TEST SET EVALUATION
  Test samples      : 9363
  Accuracy          : 99.53%
  Precision         : 99.54%
  Recall            : 99.53%
  F1 Score          : 99.53%
------------------------------------------------------------
  Per-class:
    normal       → precision: 0.99  recall: 1.00  f1: 0.99
    shoplifting  → precision: 1.00  recall: 0.99  f1: 1.00

Full classification report:
              precision    recall  f1-score   support

      normal       0.99      1.00      0.99      3205
 shoplifting       1.00      0.99      1.00      6158

    accuracy                           1.00      9363
   macro avg       0.99      1.00      0.99      9363
weighted avg       1.00      1.00      1.00      9363



<Figure size 600x500 with 2 Axes>

Confusion matrix saved to: D:\Santosh\Project_DigitalWitness\outputs\cases\mobilenet_confusion_matrix.png
Metrics saved to: D:\Santosh\Project_DigitalWitness\models\mobilenet_eval.json


## Cell 7 — BiLSTM Evaluation on Held-Out Test Set + XAI Visualisation

Evaluates the BiLSTM classifier on the held-out test sequences.
Also generates the XAI attention weight plot — showing which frames in a sample sequence drove the classification decision.

In [20]:
# ── CELL 7 — BiLSTM Evaluation (TEST SET) + XAI Attention Plot ───────────────
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ── PART A: METRICS ───────────────────────────────────────────────────────────

# Load BiLSTM test split (saved by Cell 5)
bilstm_test_path = MODELS_DIR / 'bilstm_test_split.json'
if not bilstm_test_path.exists():
    raise FileNotFoundError(
        f'BiLSTM test split not found: {bilstm_test_path}\n'
        'Run Cell 5 first to generate the test split.'
    )

with open(bilstm_test_path, 'r') as f:
    btest = json.load(f)

X_btest = [Path(p) for p in btest['paths']]
y_btest = btest['labels']
print(f'BiLSTM test split loaded: {len(X_btest)} sequences')

# Re-define model classes if running Cell 7 in isolation
class _TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, lstm_output):
        scores  = self.attn(lstm_output).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class _BiLSTMEval(nn.Module):
    def __init__(self):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=MOBILENET_FEATURE_DIM,
            hidden_size=LSTM_HIDDEN_DIM,
            num_layers=LSTM_NUM_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=LSTM_DROPOUT if LSTM_NUM_LAYERS > 1 else 0.0,
        )
        self.attention  = _TemporalAttention(LSTM_HIDDEN_DIM * 2)
        self.dropout    = nn.Dropout(LSTM_DROPOUT)
        self.classifier = nn.Linear(LSTM_HIDDEN_DIM * 2, 2)
    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        out = self.classifier(self.dropout(context))
        return out, attn_weights

if not BILSTM_SAVE.exists():
    raise FileNotFoundError(
        f'Trained BiLSTM not found: {BILSTM_SAVE}\n'
        'Run Cell 5 to train first.'
    )

bilstm_eval = _BiLSTMEval().to(device)
bckpt = torch.load(BILSTM_SAVE, map_location=device)
bilstm_eval.load_state_dict(bckpt['state_dict'])
bilstm_eval.eval()
print(f'BiLSTM loaded (best val acc: {bckpt["val_acc"]*100:.1f}%)')

class TestSeqDataset(Dataset):
    """Simple sequence dataset for evaluation — no sliding window needed."""
    def __init__(self, paths, labels, seq_len=LSTM_SEQ_LEN):
        self.paths  = paths
        self.labels = labels
        self.seq_len = seq_len
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        seq = np.load(str(self.paths[idx]))
        # Take first window or pad to seq_len
        window = seq[:self.seq_len]
        if window.shape[0] < self.seq_len:
            pad    = np.zeros((self.seq_len - window.shape[0], MOBILENET_FEATURE_DIM))
            window = np.concatenate([window, pad], axis=0)
        return torch.tensor(window, dtype=torch.float32), self.labels[idx]

btest_ds     = TestSeqDataset(X_btest, y_btest)
btest_loader = DataLoader(btest_ds, batch_size=16, shuffle=False,
                          num_workers=NUM_WORKERS)

b_preds = []
b_true  = []
b_attn_weights_list = []   # store attention weights for XAI plot
b_conf_list         = []   # store confidence scores

with torch.no_grad():
    for seqs, labels in tqdm(btest_loader, desc='Evaluating BiLSTM'):
        seqs        = seqs.to(device)
        logits, atw = bilstm_eval(seqs)
        probs        = torch.softmax(logits, dim=1)
        preds        = logits.argmax(1).cpu().numpy()
        b_preds.extend(preds.tolist())
        b_true.extend(labels.numpy().tolist())
        b_attn_weights_list.extend(atw.cpu().numpy().tolist())
        b_conf_list.extend(probs[:, 1].cpu().numpy().tolist())  # shoplifting confidence

b_acc  = accuracy_score(b_true, b_preds)
b_prec = precision_score(b_true, b_preds, average='weighted', zero_division=0)
b_rec  = recall_score(b_true, b_preds, average='weighted', zero_division=0)
b_f1   = f1_score(b_true, b_preds, average='weighted', zero_division=0)
b_cm   = confusion_matrix(b_true, b_preds)
b_cr   = classification_report(b_true, b_preds,
                                target_names=BEHAVIOR_CLASSES, zero_division=0)

print('=' * 60)
print('  BILSTM — TEST SET EVALUATION')
print('=' * 60)
print(f'  Test sequences    : {len(b_true)}')
print(f'  Accuracy          : {b_acc*100:.2f}%')
print(f'  Precision         : {b_prec*100:.2f}%')
print(f'  Recall            : {b_rec*100:.2f}%')
print(f'  F1 Score          : {b_f1*100:.2f}%')
print('-' * 60)
print('Full classification report:')
print(b_cr)

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(b_cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('BiLSTM — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
b_cm_path = OUTPUTS_DIR / 'bilstm_confusion_matrix.png'
plt.savefig(b_cm_path, dpi=150, bbox_inches='tight')
plt.show()

# Save metrics
bilstm_eval_results = {
    'model': 'BiLSTM+Attention',
    'test_sequences': len(b_true),
    'accuracy':  float(b_acc),
    'precision': float(b_prec),
    'recall':    float(b_rec),
    'f1_score':  float(b_f1),
    'confusion_matrix': b_cm.tolist(),
}
with open(MODELS_DIR / 'bilstm_eval.json', 'w') as f:
    json.dump(bilstm_eval_results, f, indent=2)
print(f'BiLSTM metrics saved to: {MODELS_DIR / "bilstm_eval.json"}')

# ── PART B: XAI ATTENTION WEIGHT VISUALISATION ────────────────────────────────

def plot_attention_weights(attention_weights, title='Temporal Attention — XAI Output',
                            predicted_class='shoplifting', confidence=0.0):
    """
    Plot attention weights as a bar chart over the sequence window.

    Each bar represents one frame in the 45-frame window.
    Bar height = attention weight (how much this frame contributed to the decision).
    Bars are colour-coded: high attention = red (suspicious), low = green (normal).

    This is the XAI explanation: 'The model classified this sequence as
    shoplifting because frames 18-24 showed the highest suspicious activity.'

    In the thesis, this figure demonstrates temporal explainability —
    the system can tell an operator *when* in the video the suspicious
    behaviour occurred, not just that it occurred.

    Parameters:
        attention_weights : list[float] — per-frame attention weights
        title             : str — chart title
        predicted_class   : str — predicted class label
        confidence        : float — prediction confidence [0, 1]
    """
    fig, ax = plt.subplots(figsize=(12, 4))
    frames = range(len(attention_weights))
    # Red = high attention (suspicious frame), green = low attention (normal frame)
    colors = ['#e74c3c' if w > 0.04 else '#2ecc71' for w in attention_weights]
    ax.bar(frames, attention_weights, color=colors, edgecolor='white', linewidth=0.5)
    # Dashed line showing what uniform attention would look like
    ax.axhline(1 / len(attention_weights), color='gray', linestyle='--', linewidth=1,
               label=f'Uniform baseline (1/{len(attention_weights):.0f})')
    ax.set_xlabel('Frame index in 45-frame window (7.5 seconds @ 6fps)', fontsize=11)
    ax.set_ylabel('Attention weight', fontsize=11)
    ax.set_title(f'{title}\nPredicted: {predicted_class.upper()}  |  '
                 f'Confidence: {confidence:.1%}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    xai_path = OUTPUTS_DIR / 'attention_xai_example.png'
    plt.savefig(xai_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'XAI attention plot saved to: {xai_path}')
    return fig

# Find a high-confidence shoplifting prediction for the XAI example
xai_idx = None
for i, (pred, true_lbl, conf) in enumerate(zip(b_preds, b_true, b_conf_list)):
    if pred == 1 and conf > 0.7:   # shoplifting, confidence > 70%
        xai_idx = i
        break

if xai_idx is not None:
    attn_w = b_attn_weights_list[xai_idx]
    plot_attention_weights(
        attn_w,
        title='Temporal Attention — XAI Output (BiLSTM)',
        predicted_class='shoplifting',
        confidence=b_conf_list[xai_idx],
    )
else:
    # Fall back to any shoplifting prediction if no high-confidence one exists
    for i, pred in enumerate(b_preds):
        if pred == 1:
            xai_idx = i
            break
    if xai_idx is not None:
        plot_attention_weights(b_attn_weights_list[xai_idx],
                               predicted_class='shoplifting',
                               confidence=b_conf_list[xai_idx])
    else:
        print('No shoplifting predictions in test set — XAI plot not generated.')

C:\Users\johnf\AppData\Local\Temp\ipykernel_5420\1370933462.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bckpt = torch.load(BILSTM_SAVE, map_location=device)


BiLSTM test split loaded: 27 sequences
BiLSTM loaded (best val acc: 100.0%)


Evaluating BiLSTM: 100%|██████████| 2/2 [00:00<00:00,  5.81it/s]


  BILSTM — TEST SET EVALUATION
  Test sequences    : 27
  Accuracy          : 96.30%
  Precision         : 96.56%
  Recall            : 96.30%
  F1 Score          : 96.30%
------------------------------------------------------------
Full classification report:
              precision    recall  f1-score   support

      normal       0.93      1.00      0.96        13
 shoplifting       1.00      0.93      0.96        14

    accuracy                           0.96        27
   macro avg       0.96      0.96      0.96        27
weighted avg       0.97      0.96      0.96        27



<Figure size 600x500 with 2 Axes>

BiLSTM metrics saved to: D:\Santosh\Project_DigitalWitness\models\bilstm_eval.json


<Figure size 1200x400 with 1 Axes>

XAI attention plot saved to: D:\Santosh\Project_DigitalWitness\outputs\cases\attention_xai_example.png


### XAI Interpretation

The bar chart above shows the temporal attention weights produced by the BiLSTM for a single 45-frame (7.5-second) window classified as shoplifting. Each bar height represents the proportion of the model's decision attributed to that specific frame. Frames with weights above the uniform baseline (dashed line) indicate moments of heightened suspicious activity — typically the frames in which concealment gestures or product interactions occurred. This mechanism provides *temporal explainability*: rather than simply reporting a binary classification, the system identifies **when** in the sequence the suspicious behaviour was most prominent, enabling operators to review only the flagged time segment rather than the entire video.

## Cell 8 — PersonProductTracker and POS Cross-Checking

Reads YOLO class detections per tracked person and builds a product-state summary. This is then cross-referenced with POS transaction data to detect discrepancies (items detected but not billed).

**How product detection works without VLMs:**
The 24-class YOLO schema encodes product state directly in the class label. `person-with-filled-basket-trolly` (class 21) means YOLO detected a person carrying a filled basket — no separate product detection needed. The transition from class 21 → class 19 (`person-with-empty-basket-trolly`) without a visit to `occupied-checkout-counter` (class 8) is a potential concealment or bypass event.

**Academic justification for mock POS:**
Real POS systems expose transaction data via pole display APIs or receipt printer data streams (Venetianer et al., 2007; RetailWatch architecture). Our mock POS generates structurally identical JSON events: `{session_id, timestamp, items_billed, transaction_total}`. A production integration would replace the mock generator with a POS API adapter — the fusion logic is POS-agnostic by design.

In [21]:
# ── CELL 8 — PersonProductTracker + POS Integration ──────────────────────────
import datetime
import random
import uuid

class PersonProductTracker:
    """
    Tracks behaviour-related state for a single person across video frames.

    Updated every frame with the YOLO class IDs detected for this person.
    Works with the 4-class behaviour dataset:
        0: Looking around  — suspicious reconnaissance
        1: Picking-Holding — product interaction / concealment gesture
        2: normal          — normal behaviour
        3: shoplifting     — direct shoplifting detection

    State tracked:
        direct_shoplifting_frames : frames where class 3 (shoplifting) detected
        looking_around_frames     : frames where class 0 (Looking around) detected
        picking_holding_frames    : frames where class 1 (Picking-Holding) detected
        normal_frames             : frames where class 2 (normal) detected
        max_products_held         : peak count of PRODUCT_HELD_IDS detected
        concealment_frames        : frames with CONCEALMENT_IDS (classes 0 + 3)
        checkout_visited          : always False (not in this schema)
        product_frames            : frames with product interaction (class 1)
        total_frames              : total frames this person was tracked
    """
    def __init__(self, track_id):
        self.track_id                  = track_id
        self.direct_shoplifting_frames = 0
        self.looking_around_frames     = 0
        self.picking_holding_frames    = 0
        self.normal_frames             = 0
        self.max_products_held         = 0
        self.concealment_frames        = 0
        self.checkout_visited          = False   # not in 4-class schema
        self.product_frames            = 0
        self.total_frames              = 0
        self._class_history            = []

    def update(self, detected_class_ids: set, checkout_nearby: bool = False):
        """
        Called once per frame with the set of class IDs detected for this person.

        With the 4-class schema, direct shoplifting detection (class 3) and
        looking-around behaviour (class 0) are the strongest intent signals.
        Picking-Holding (class 1) provides product interaction context.

        Why track direct_shoplifting_frames separately?
        Unlike the 24-class schema where shoplifting was inferred from object
        states, this dataset has a direct 'shoplifting' class — a frame-level
        behaviour label annotated by human reviewers. Counting these frames
        separately gives the intent scorer a high-confidence direct signal.

        Parameters:
            detected_class_ids : set[int] — YOLO class IDs in this person's bbox
            checkout_nearby    : bool — not used in 4-class schema, kept for API compat
        """
        self.total_frames += 1
        self._class_history.append(set(detected_class_ids))

        # Direct shoplifting detection — highest confidence signal
        if 3 in detected_class_ids:
            self.direct_shoplifting_frames += 1

        # Suspicious reconnaissance behaviour
        if 0 in detected_class_ids:
            self.looking_around_frames += 1

        # Product interaction frames
        if 1 in detected_class_ids:
            self.picking_holding_frames += 1
            self.product_frames += 1
            self.max_products_held = max(self.max_products_held,
                                         len(detected_class_ids & PRODUCT_HELD_IDS))

        # Normal behaviour frames
        if 2 in detected_class_ids:
            self.normal_frames += 1

        # Concealment = Looking around (reconnaissance) + direct shoplifting
        if detected_class_ids & CONCEALMENT_IDS:
            self.concealment_frames += 1

    def summary(self) -> dict:
        """
        Return a summary dict for the intent scorer and POS comparator.

        Returns:
            dict with keys: track_id, max_products_held, concealment_frames,
            checkout_visited, product_frames, total_frames,
            concealment_ratio, product_ratio,
            direct_shoplifting_frames, shoplifting_ratio,
            looking_around_frames, picking_holding_frames
        """
        total = max(self.total_frames, 1)
        return {
            'track_id':                    self.track_id,
            'max_products_held':           self.max_products_held,
            'concealment_frames':          self.concealment_frames,
            'checkout_visited':            self.checkout_visited,
            'product_frames':              self.product_frames,
            'total_frames':                self.total_frames,
            'concealment_ratio':           self.concealment_frames          / total,
            'product_ratio':               self.product_frames              / total,
            'direct_shoplifting_frames':   self.direct_shoplifting_frames,
            'shoplifting_ratio':           self.direct_shoplifting_frames   / total,
            'looking_around_frames':       self.looking_around_frames,
            'picking_holding_frames':      self.picking_holding_frames,
        }



class MockPOSDatabase:
    """
    Simulates a POS transaction database for testing.

    In a production deployment this class would be replaced by an API client
    connecting to the store's POS system (e.g. via the pole display data stream
    or receipt printer API described in Venetianer et al., 2007).

    The mock generates transactions with realistic timestamps and item counts,
    including one pre-configured suspicious session where fewer items are billed
    than detected by YOLO.
    """
    def __init__(self):
        import datetime
        now = datetime.datetime.now()
        self.transactions = {
            'SESS_001': {
                'session_id':        'SESS_001',
                'timestamp':         (now - datetime.timedelta(minutes=30)).isoformat(),
                'items_billed':      4,
                'transaction_total': 22.50,
                'status':            'complete',
            },
            'SESS_002': {
                'session_id':        'SESS_002',
                'timestamp':         (now - datetime.timedelta(minutes=15)).isoformat(),
                'items_billed':      1,    # suspicious: YOLO detects more items
                'transaction_total': 3.99,
                'status':            'complete',
            },
            'SESS_003': {
                'session_id':        'SESS_003',
                'timestamp':         now.isoformat(),
                'items_billed':      0,    # bypass: no transaction at all
                'transaction_total': 0.0,
                'status':            'no_transaction',
            },
        }

    def get_transaction(self, session_id: str):
        """Look up a transaction by session ID. Returns None if not found."""
        return self.transactions.get(session_id)

    def match_by_timestamp(self, video_timestamp: str, window_seconds: int = 300):
        """Find the most recent transaction within window_seconds of the video timestamp."""
        import datetime
        try:
            vt = datetime.datetime.fromisoformat(video_timestamp)
        except Exception:
            return None
        best, best_delta = None, float('inf')
        for tx in self.transactions.values():
            try:
                tt    = datetime.datetime.fromisoformat(tx['timestamp'])
                delta = abs((vt - tt).total_seconds())
                if delta < window_seconds and delta < best_delta:
                    best, best_delta = tx, delta
            except Exception:
                continue
        return best


def compare_with_pos(person_summary: dict, pos_transaction: dict) -> dict:
    """
    Cross-reference YOLO-detected items with POS transaction data.

    With the 4-class schema, 'items detected' = max_products_held
    (frames where Picking-Holding class was detected).

    Returns:
        dict with keys: discrepancy_count, discrepancy_ratio, flag_type,
        requires_review, items_detected, items_billed
    """
    items_detected  = person_summary.get('max_products_held', 0)
    checkout_ok     = person_summary.get('checkout_visited', False)

    if pos_transaction is None:
        flag_type = 'bypass' if not checkout_ok else 'lookup_failed'
        return {
            'items_detected':    items_detected,
            'items_billed':      0,
            'discrepancy_count': items_detected,
            'discrepancy_ratio': 1.0,
            'flag_type':         flag_type,
            'requires_review':   items_detected > 0,
        }

    items_billed   = pos_transaction.get('items_billed', 0)
    discrepancy    = max(0, items_detected - items_billed)
    disc_ratio     = discrepancy / max(1, items_detected)

    if discrepancy == 0:
        flag_type = 'clean'
    elif disc_ratio < 0.25:
        flag_type = 'minor_mismatch'
    elif disc_ratio < 0.60:
        flag_type = 'suspicious'
    else:
        flag_type = 'bypass'

    return {
        'items_detected':    items_detected,
        'items_billed':      items_billed,
        'discrepancy_count': discrepancy,
        'discrepancy_ratio': float(disc_ratio),
        'flag_type':         flag_type,
        'requires_review':   discrepancy > 0,
    }

# ── DEMO: Show how the 4-class tracker works ──────────────────────────────────
print('PersonProductTracker + POS Demo (4-class behaviour schema)')
print('-' * 50)

# Simulate a person: normal → looking around → picking/holding → shoplifting
tracker = PersonProductTracker(track_id=1)
# Frames 1-10: normal shopping
for _ in range(10):
    tracker.update({2})           # normal
# Frames 11-15: starts looking around suspiciously
for _ in range(5):
    tracker.update({0})           # Looking around
# Frames 16-25: picks up item and conceals it
for _ in range(10):
    tracker.update({1})           # Picking-Holding
# Frames 26-30: shoplifting detected
for _ in range(5):
    tracker.update({3})           # shoplifting

summary = tracker.summary()
print(f'Tracker summary:')
for k, v in summary.items():
    print(f'  {k:<32}: {v}')

# POS cross-check: person picked up 1 item, billed 0
pos_db = MockPOSDatabase()
tx     = pos_db.get_transaction('SESS_002')   # suspicious session (1 item billed)
result = compare_with_pos(summary, tx)
print(f'\nPOS comparison: {json.dumps(result, indent=2)}')
print(f'\nFlag: {result["flag_type"].upper()} — '
      f'detected {result["items_detected"]} item(s), '
      f'billed {result["items_billed"]}')
print(f'\nDirect shoplifting frames: {summary["direct_shoplifting_frames"]} / '
      f'{summary["total_frames"]} '
      f'({summary["shoplifting_ratio"]*100:.0f}% of tracked time)')

PersonProductTracker + POS Demo (4-class behaviour schema)
--------------------------------------------------
Tracker summary:
  track_id                        : 1
  max_products_held               : 1
  concealment_frames              : 10
  checkout_visited                : False
  product_frames                  : 10
  total_frames                    : 30
  concealment_ratio               : 0.3333333333333333
  product_ratio                   : 0.3333333333333333
  direct_shoplifting_frames       : 5
  shoplifting_ratio               : 0.16666666666666666
  looking_around_frames           : 5
  picking_holding_frames          : 10

POS comparison: {
  "items_detected": 1,
  "items_billed": 1,
  "discrepancy_count": 0,
  "discrepancy_ratio": 0.0,
  "flag_type": "clean",
  "requires_review": false
}

Flag: CLEAN — detected 1 item(s), billed 1

Direct shoplifting frames: 5 / 30 (17% of tracked time)


## Cell 9 — Full Inference Pipeline

Runs end-to-end analysis on a single video:
1. YOLO26n detects persons and retail objects, ByteTrack assigns IDs
2. For each tracked person, crop is fed through MobileNetV2
3. Feature sequences fed through BiLSTM → classification + attention weights
4. PersonProductTracker updated from YOLO class IDs
5. Results aggregated with sliding-window weighted voting

In [22]:
# ── CELL 9 — Full Inference Pipeline ──────────────────────────────────────────
import cv2
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict, deque
from pathlib import Path
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights
from torchvision import models as tv_models

# Lazy-load models — only load if not already in memory from earlier cells
def _load_mobilenet(path, device):
    """Load MobileNetV2 from checkpoint, return in eval mode."""
    class _MNet(nn.Module):
        def __init__(self):
            super().__init__()
            base = tv_models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
            self.features   = base.features
            self.pool       = nn.AdaptiveAvgPool2d((1, 1))
            self.classifier = nn.Sequential(
                nn.Linear(MOBILENET_FEATURE_DIM, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(512, 2),
            )
        def forward(self, x):
            x = self.features(x); x = self.pool(x)
            return self.classifier(torch.flatten(x, 1))
        def extract_features(self, x):
            return torch.flatten(self.pool(self.features(x)), 1)

    m = _MNet().to(device)
    ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt['state_dict'])
    m.eval()
    return m

def _load_bilstm(path, device):
    """Load BiLSTM from checkpoint, return in eval mode."""
    class _Attn(nn.Module):
        def __init__(self, h):
            super().__init__()
            self.attn = nn.Sequential(nn.Linear(h, h//2), nn.Tanh(), nn.Linear(h//2, 1))
        def forward(self, x):
            s = self.attn(x).squeeze(-1)
            w = torch.softmax(s, dim=1)
            return torch.bmm(w.unsqueeze(1), x).squeeze(1), w

    class _BLSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.bilstm = nn.LSTM(MOBILENET_FEATURE_DIM, LSTM_HIDDEN_DIM,
                                   LSTM_NUM_LAYERS, batch_first=True,
                                   bidirectional=True,
                                   dropout=LSTM_DROPOUT if LSTM_NUM_LAYERS > 1 else 0.0)
            self.attention  = _Attn(LSTM_HIDDEN_DIM * 2)
            self.dropout    = nn.Dropout(LSTM_DROPOUT)
            self.classifier = nn.Linear(LSTM_HIDDEN_DIM * 2, 2)
        def forward(self, x):
            out, _ = self.bilstm(x)
            ctx, w = self.attention(out)
            return self.classifier(self.dropout(ctx)), w

    m = _BLSTM().to(device)
    ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt['state_dict'])
    m.eval()
    return m


_infer_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(MOBILENET_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def run_inference(video_path, yolo_path=None, mobilenet_path=None,
                  bilstm_path=None, frame_step=1, progress_cb=None):
    """
    Full inference pipeline on a single video file.

    Steps:
        1. YOLO detects objects and assigns ByteTrack IDs per frame
        2. For each tracked person, crop through MobileNetV2 feature extractor
        3. Sliding BiLSTM windows classify feature sequences
        4. PersonProductTracker updated with YOLO class IDs per frame
        5. Weighted voting aggregates window predictions into final result

    Parameters:
        video_path    : str or Path — input video file
        yolo_path     : Path — fine-tuned YOLO weights (default: YOLO26_RETAIL)
        mobilenet_path: Path — MobileNetV2 checkpoint (default: MOBILENET_SAVE)
        bilstm_path   : Path — BiLSTM checkpoint (default: BILSTM_SAVE)
        frame_step    : int — process every Nth frame (3 = 3x faster, less accurate)
        progress_cb   : callable(fraction, message) — optional UI progress callback

    Returns:
        dict with keys: overall_class, overall_conf, is_shoplifting,
        predictions, attention_maps, person_summaries, detections, behavior_events
    """
    yolo_path      = yolo_path      or YOLO26_RETAIL
    mobilenet_path = mobilenet_path or MOBILENET_SAVE
    bilstm_path    = bilstm_path    or BILSTM_SAVE

    # Verify model files exist before loading
    for p, name in [(yolo_path, 'YOLO'), (mobilenet_path, 'MobileNetV2'),
                    (bilstm_path, 'BiLSTM')]:
        if not Path(p).exists():
            raise FileNotFoundError(
                f'{name} model not found: {p}\n'
                'Run the corresponding training cell first.'
            )

    # Load models
    from ultralytics import YOLO as _YOLO
    yolo_model    = _YOLO(str(yolo_path))
    mnet_model    = _load_mobilenet(mobilenet_path, device)
    bilstm_model  = _load_bilstm(bilstm_path, device)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_video    = cap.get(cv2.CAP_PROP_FPS) or 25.0

    # Per-person state
    trackers          = {}     # track_id → PersonProductTracker
    feature_buffers   = defaultdict(list)   # track_id → deque of 1280-dim vectors
    predictions_list  = []
    attention_maps    = defaultdict(list)
    behavior_events   = []

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Skip frames according to frame_step for faster processing
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        timestamp = frame_idx / fps_video

        # ── YOLO detection with ByteTrack tracking ─────────────────────────────
        results    = yolo_model.track(frame, persist=True, verbose=False,
                                       tracker='bytetrack.yaml')
        detections = results[0].boxes if results else None

        checkout_detected = False

        if detections is not None and detections.id is not None:
            ids    = detections.id.cpu().numpy().astype(int)
            cls_ids = detections.cls.cpu().numpy().astype(int)
            boxes  = detections.xyxy.cpu().numpy().astype(int)

            # Check if checkout counter is in this frame
            if CHECKOUT_OCCUPIED_ID in cls_ids:
                checkout_detected = True

            for det_idx, (tid, cid, box) in enumerate(zip(ids, cls_ids, boxes)):
                # Update PersonProductTracker for this track ID
                if tid not in trackers:
                    trackers[tid] = PersonProductTracker(track_id=tid)
                trackers[tid].update({cid}, checkout_nearby=checkout_detected)

                # Only extract MobileNetV2 features for person detections
                if cid in PERSON_CLASS_IDS:
                    x1, y1, x2, y2 = box
                    # Clamp crop to frame boundaries
                    x1 = max(0, x1); y1 = max(0, y1)
                    x2 = min(frame.shape[1], x2); y2 = min(frame.shape[0], y2)
                    crop = frame[y1:y2, x1:x2]

                    if crop.size > 0:
                        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                        tensor   = _infer_transform(crop_rgb).unsqueeze(0).to(device)
                        with torch.no_grad():
                            feat = mnet_model.extract_features(tensor)
                        feature_buffers[tid].append(feat.cpu().numpy()[0])

                        # Run BiLSTM when we have enough features for a full window
                        if len(feature_buffers[tid]) >= LSTM_SEQ_LEN:
                            window = np.array(feature_buffers[tid][-LSTM_SEQ_LEN:])
                            seq_t  = torch.tensor(window, dtype=torch.float32
                                                   ).unsqueeze(0).to(device)
                            with torch.no_grad():
                                logits, attn_w = bilstm_model(seq_t)
                                probs          = torch.softmax(logits, dim=1)
                            pred_cls  = logits.argmax(1).item()
                            shop_conf = probs[0, 1].item()

                            predictions_list.append({
                                'timestamp':    timestamp,
                                'track_id':     int(tid),
                                'prediction':   BEHAVIOR_CLASSES[pred_cls],
                                'confidence':   float(shop_conf),
                                'is_shoplifting': pred_cls == 1,
                            })
                            attention_maps[int(tid)].append(
                                attn_w.squeeze(0).cpu().numpy().tolist()
                            )

                            if pred_cls == 1:
                                behavior_events.append({
                                    'type':      'shoplifting_window',
                                    'timestamp': timestamp,
                                    'track_id':  int(tid),
                                    'confidence': float(shop_conf),
                                })

        frame_idx += 1

        # Optional progress callback for UI integration
        if progress_cb and total_frames > 0:
            progress_cb(frame_idx / total_frames,
                        f'Processing frame {frame_idx}/{total_frames}')

    cap.release()

    # ── SLIDING WINDOW WEIGHTED VOTING ────────────────────────────────────────
    # Each window produces one prediction. Shoplifting windows get 2x weight
    # (conservative: prefer to flag for human review over missing real theft).
    # Final classification requires shop_score > 0.3 of total score.
    shop_score  = 0.0
    total_score = 0.0

    for p in predictions_list:
        weight = 2.0 if p['is_shoplifting'] else 1.0
        total_score += weight
        if p['is_shoplifting']:
            shop_score += p['confidence'] * weight

    if total_score > 0:
        overall_conf     = shop_score / total_score
        is_shoplifting   = (shop_score / total_score) > 0.3
    else:
        overall_conf   = 0.0
        is_shoplifting = False

    overall_class = 'shoplifting' if is_shoplifting else 'normal'

    return {
        'overall_class':    overall_class,
        'overall_conf':     float(overall_conf),
        'is_shoplifting':   is_shoplifting,
        'predictions':      predictions_list,
        'attention_maps':   dict(attention_maps),
        'person_summaries': {tid: t.summary() for tid, t in trackers.items()},
        'detections': {
            'persons_tracked':  len(trackers),
            'frames_processed': frame_idx,
            'duration':         frame_idx / fps_video,
        },
        'behavior_events': behavior_events,
    }

print('run_inference() defined. Call it with:')
print('  results = run_inference("path/to/video.mp4")')

run_inference() defined. Call it with:
  results = run_inference("path/to/video.mp4")


## Cell 10 — Intent Scoring and Bias-Aware Assessment

Converts raw model outputs into a structured, interpretable risk score.
This is the core research contribution: decomposed, explainable scoring with a bias-aware quality adjustment.

**Intent score formula:**
```
score = W_BEHAVIOUR   × behaviour_score    (0.40)
       + W_CONCEALMENT × concealment_score (0.30)
       + W_POS_MISMATCH × pos_score        (0.20)
       + W_DURATION    × duration_score    (0.10)
```
Each component is independently computed and reported, so an operator can see exactly **why** a particular score was generated — not just the final number. This addresses the 'black box' problem identified in Jebur et al. (2025) and Kim et al. (2021).

In [23]:
# ── CELL 10 — Intent Scoring + Bias-Aware Assessment ─────────────────────────

def calculate_intent_score(inference_result: dict, pos_comparison: dict = None) -> dict:
    """
    Compute a decomposed intent score from inference result components.

    Formula:
        score = W_BEHAVIOUR   × behaviour_score    # BiLSTM output
               + W_CONCEALMENT × concealment_score # YOLO concealment classes
               + W_POS_MISMATCH × pos_score        # POS discrepancy
               + W_DURATION    × duration_score    # temporal persistence

    Why decomposed scoring?
    A single number (e.g. 'score=0.72') provides no actionable information.
    The decomposed report tells the operator: 'high score primarily from
    concealment detection (0.30/0.30) with moderate behaviour signal (0.26/0.40)
    and POS mismatch (0.14/0.20)'. This supports human decision-making and
    provides an audit trail — a key requirement identified by Archana et al. (2024).

    Parameters:
        inference_result : dict — output of run_inference()
        pos_comparison   : dict — output of compare_with_pos() or None

    Returns:
        dict with final_score, risk_level, components, and flag details
    """
    predictions    = inference_result.get('predictions', [])
    person_sums    = inference_result.get('person_summaries', {})

    # ── COMPONENT 1: Behaviour score ─────────────────────────────────────────
    # Mean shoplifting confidence across all BiLSTM windows
    shop_confs = [p['confidence'] for p in predictions if p['is_shoplifting']]
    all_confs  = [p['confidence'] for p in predictions]

    if all_confs:
        # Weight by shoplifting windows: average shop confidence if any, else 0
        behaviour_score = float(np.mean(shop_confs)) if shop_confs else 0.0
    else:
        behaviour_score = 0.0

    # ── COMPONENT 2: Concealment score ───────────────────────────────────────
    # Maximum concealment ratio across all tracked persons
    concealment_ratios = [
        s['concealment_ratio'] for s in person_sums.values()
    ]
    concealment_score = float(max(concealment_ratios)) if concealment_ratios else 0.0

    # ── COMPONENT 3: POS mismatch score ──────────────────────────────────────
    if pos_comparison:
        pos_score = float(pos_comparison['discrepancy_ratio'])
        pos_score = min(1.0, pos_score)   # cap at 1.0
    else:
        pos_score = 0.0   # no POS data available

    # ── COMPONENT 4: Duration score ───────────────────────────────────────────
    # Proportion of tracked time spent in suspicious state (product interaction)
    product_ratios = [s['product_ratio'] for s in person_sums.values()]
    duration_score = float(max(product_ratios)) if product_ratios else 0.0

    # ── WEIGHTED SUM ──────────────────────────────────────────────────────────
    final_score = (
        W_BEHAVIOUR    * behaviour_score   +
        W_CONCEALMENT  * concealment_score +
        W_POS_MISMATCH * pos_score         +
        W_DURATION     * duration_score
    )

    # ── RISK LEVEL CLASSIFICATION ─────────────────────────────────────────────
    if final_score >= THRESHOLD_CRITICAL:
        risk_level = 'CRITICAL'
    elif final_score >= THRESHOLD_HIGH:
        risk_level = 'HIGH'
    elif final_score >= THRESHOLD_MEDIUM:
        risk_level = 'MEDIUM'
    elif final_score >= THRESHOLD_LOW:
        risk_level = 'LOW'
    else:
        risk_level = 'MINIMAL'

    return {
        'final_score':       float(final_score),
        'risk_level':        risk_level,
        'components': {
            'behaviour':   float(behaviour_score),
            'concealment': float(concealment_score),
            'pos_mismatch': float(pos_score),
            'duration':    float(duration_score),
        },
        'weighted_components': {
            'behaviour':    float(W_BEHAVIOUR    * behaviour_score),
            'concealment':  float(W_CONCEALMENT  * concealment_score),
            'pos_mismatch': float(W_POS_MISMATCH * pos_score),
            'duration':     float(W_DURATION     * duration_score),
        },
        'pos_available': pos_comparison is not None,
    }


def bias_aware_adjustment(intent_score_dict: dict, quality_score: float = 1.0) -> dict:
    """
    Apply a quality-based fairness adjustment to the intent score.

    This function operationalises the bias-aware principle:
    When evidence quality is low (poor video, partial occlusion, brief
    observation window), we reduce confidence rather than maintain a
    potentially misleading high score.

    quality_score parameter:
        1.0 = ideal conditions (good lighting, clear YOLO detections)
        0.75 = moderate quality (some occlusion, partial tracking)
        0.5  = poor quality (low light, frequent tracking loss)
        < 0.4 = analysis unreliable — flag for human review regardless of score

    Academic note: This mechanism addresses the quality-based subset of
    algorithmic bias. It does not measure demographic disparate impact
    (which requires protected attribute data not available in this dataset),
    but it does ensure the system does not generate high-confidence alerts
    from low-quality evidence — a key source of false positives and
    wrongful accusations in surveillance systems (Archana et al., 2024).

    Parameters:
        intent_score_dict : dict — output of calculate_intent_score()
        quality_score     : float in [0, 1] — evidence quality

    Returns:
        dict — adjusted intent score with quality metadata
    """
    original_score  = intent_score_dict['final_score']
    adjusted_score  = original_score * quality_score

    # If quality is very low, force human review regardless of score
    force_review = quality_score < 0.4

    # Recalculate risk level with adjusted score
    if force_review or adjusted_score >= THRESHOLD_CRITICAL:
        risk_level = 'CRITICAL' if not force_review else 'REVIEW_REQUIRED'
    elif adjusted_score >= THRESHOLD_HIGH:
        risk_level = 'HIGH'
    elif adjusted_score >= THRESHOLD_MEDIUM:
        risk_level = 'MEDIUM'
    elif adjusted_score >= THRESHOLD_LOW:
        risk_level = 'LOW'
    else:
        risk_level = 'MINIMAL'

    result = dict(intent_score_dict)
    result.update({
        'original_score':    float(original_score),
        'final_score':       float(adjusted_score),
        'quality_score':     float(quality_score),
        'risk_level':        risk_level,
        'force_human_review': force_review,
        'bias_note': (
            'Score adjusted for evidence quality. '
            'Low-quality evidence should not drive high-confidence alerts.'
        ),
    })
    return result


def detect_edge_cases(inference_result: dict, intent_score: dict) -> list:
    """
    Detect edge cases that warrant special handling or review.

    Edge cases include: very brief tracking duration, no person detected,
    very high concealment with low behaviour score (possible false positive),
    and single-frame anomalies.

    Returns: list of edge case strings
    """
    edge_cases = []
    detections    = inference_result.get('detections', {})
    person_sums   = inference_result.get('person_summaries', {})
    components    = intent_score.get('components', {})

    if detections.get('persons_tracked', 0) == 0:
        edge_cases.append('NO_PERSON_DETECTED')

    if detections.get('duration', 0) < 5.0:
        edge_cases.append('BRIEF_OBSERVATION_WINDOW')

    # High concealment but low behaviour score may indicate occlusion artefacts
    if (components.get('concealment', 0) > 0.5 and
            components.get('behaviour', 0) < 0.2):
        edge_cases.append('CONCEALMENT_WITHOUT_BEHAVIOUR_SIGNAL')

    if not inference_result.get('predictions'):
        edge_cases.append('NO_BILSTM_WINDOWS_PROCESSED')

    return edge_cases


def generate_alert(intent_score: dict, edge_cases: list) -> dict:
    """
    Generate a structured alert from the intent score.

    Alert levels:
        NO_ALERT       : score < THRESHOLD_LOW or no person detected
        ADVISORY       : LOW risk — log but do not notify
        ALERT          : MEDIUM/HIGH risk — notify supervisor
        CRITICAL_ALERT : CRITICAL risk — immediate response

    All alerts include a mandatory human-review disclaimer per best practice
    in automated surveillance systems (Jebur et al., 2025).

    Parameters:
        intent_score : dict — bias-adjusted intent score
        edge_cases   : list — from detect_edge_cases()

    Returns:
        dict with alert_level, message, action, and disclaimer
    """
    score     = intent_score['final_score']
    risk      = intent_score['risk_level']
    forced    = intent_score.get('force_human_review', False)

    if 'NO_PERSON_DETECTED' in edge_cases or score < THRESHOLD_LOW:
        level   = 'NO_ALERT'
        message = 'No suspicious activity detected.'
        action  = 'none'
    elif forced or risk == 'REVIEW_REQUIRED':
        level   = 'REVIEW_REQUIRED'
        message = 'Low-quality evidence — human review required before any action.'
        action  = 'flag_for_review'
    elif risk == 'CRITICAL':
        level   = 'CRITICAL_ALERT'
        message = f'CRITICAL: Intent score {score:.2f}. Immediate supervisor review required.'
        action  = 'notify_supervisor_immediately'
    elif risk == 'HIGH':
        level   = 'ALERT'
        message = f'HIGH RISK: Intent score {score:.2f}. Supervisor notification recommended.'
        action  = 'notify_supervisor'
    elif risk in ('MEDIUM', 'LOW'):
        level   = 'ADVISORY'
        message = f'Advisory: Intent score {score:.2f}. Log and monitor.'
        action  = 'log_and_monitor'
    else:
        level   = 'NO_ALERT'
        message = 'No suspicious activity detected.'
        action  = 'none'

    return {
        'alert_level':  level,
        'risk_level':   risk,
        'intent_score': float(score),
        'message':      message,
        'action':       action,
        'edge_cases':   edge_cases,
        'disclaimer':   (
            'IMPORTANT: This is an automated alert. No action should be taken '
            'based solely on this system output. All alerts must be reviewed '
            'by a trained human operator before any intervention.'
        ),
    }

print('Intent scoring functions defined:')
print('  calculate_intent_score(inference_result, pos_comparison=None)')
print('  bias_aware_adjustment(intent_score_dict, quality_score=1.0)')
print('  detect_edge_cases(inference_result, intent_score)')
print('  generate_alert(intent_score, edge_cases)')

Intent scoring functions defined:
  calculate_intent_score(inference_result, pos_comparison=None)
  bias_aware_adjustment(intent_score_dict, quality_score=1.0)
  detect_edge_cases(inference_result, intent_score)
  generate_alert(intent_score, edge_cases)


## Cell 11 — Forensic Case File and Visualisation

Builds a structured JSON case file as the audit trail for each analysis. Also generates the behaviour timeline plot and formatted results display.

The case file is the academic contribution to forensic accountability. No reviewed paper generates a structured per-incident audit trail as a first-class model output (confirmed by Perplexity literature search, March 2026). It contains: video metadata, per-window LSTM predictions, intent score components, bias flags, POS comparison result, alert ID, and a mandatory human-review disclaimer.

In [24]:
# ── CELL 11 — Case File + Visualisation ──────────────────────────────────────
import datetime
import json
import uuid
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


def build_case_file(video_path, inference_result: dict, intent_score: dict,
                    alert: dict, pos_comparison: dict = None) -> dict:
    """
    Build a structured forensic case file for the analysis.

    The case file serves as the audit trail required by GDPR Article 22
    (automated decision-making) and is the primary academic contribution
    to forensic accountability in retail AI surveillance.

    Structure:
        case_id         : unique identifier (UUID4)
        created_at      : ISO timestamp
        video_metadata  : path, duration, frames processed
        predictions     : all BiLSTM window predictions with timestamps
        intent_score    : decomposed scoring with components
        alert           : alert level, message, and human-review disclaimer
        pos_comparison  : POS cross-check result (if available)
        attention_maps  : reference to XAI attention weight data
        bias_flags      : edge cases and quality adjustments applied

    Parameters:
        video_path       : str or Path
        inference_result : dict — from run_inference()
        intent_score     : dict — from bias_aware_adjustment()
        alert            : dict — from generate_alert()
        pos_comparison   : dict — from compare_with_pos() or None

    Returns:
        dict — the complete case file
    """
    case_id = str(uuid.uuid4())
    case = {
        'case_id':      case_id,
        'created_at':   datetime.datetime.now().isoformat(),
        'video_metadata': {
            'path':             str(video_path),
            'duration_seconds': inference_result['detections'].get('duration', 0),
            'frames_processed': inference_result['detections'].get('frames_processed', 0),
            'persons_tracked':  inference_result['detections'].get('persons_tracked', 0),
        },
        'overall_result': {
            'classification': inference_result['overall_class'],
            'confidence':     inference_result['overall_conf'],
            'is_shoplifting': inference_result['is_shoplifting'],
        },
        'predictions':    inference_result.get('predictions', []),
        'person_summaries': inference_result.get('person_summaries', {}),
        'intent_score':   intent_score,
        'alert':          alert,
        'pos_comparison': pos_comparison,
        'attention_maps': {
            'available': bool(inference_result.get('attention_maps')),
            'track_ids': list(inference_result.get('attention_maps', {}).keys()),
        },
        'bias_flags': {
            'quality_score':     intent_score.get('quality_score', 1.0),
            'force_human_review': intent_score.get('force_human_review', False),
            'edge_cases':        alert.get('edge_cases', []),
        },
        'disclaimer': alert['disclaimer'],
    }
    return case


def save_case_file(case: dict) -> Path:
    """
    Save case file as JSON to the outputs/cases/ directory.

    File naming: case_{case_id[:8]}_{timestamp}.json
    This ensures unique filenames even for concurrent analyses.
    """
    ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    name = f'case_{case["case_id"][:8]}_{ts}.json'
    path = OUTPUTS_DIR / name
    with open(path, 'w') as f:
        json.dump(case, f, indent=2, default=str)
    return path


def plot_behavior_timeline(predictions: list, title: str = 'Behaviour Timeline',
                            save_path: Path = None):
    """
    Generate a Gantt-style timeline of behaviour predictions per track ID.

    Each horizontal bar represents one BiLSTM window prediction.
    Colour coding: green = normal, red = shoplifting.
    Bar width = window duration (LSTM_SEQ_LEN / FPS_TARGET seconds).

    This visualisation allows an operator to see exactly when and for
    whom suspicious behaviour was detected without reviewing the raw video.

    Parameters:
        predictions : list — from inference_result['predictions']
        title       : str — plot title
        save_path   : Path — if provided, save the figure here
    """
    if not predictions:
        print('No predictions to plot.')
        return

    # Group predictions by track ID
    by_track = {}
    for p in predictions:
        tid = p['track_id']
        by_track.setdefault(tid, []).append(p)

    fig, ax = plt.subplots(figsize=(14, max(3, len(by_track) * 1.5)))
    window_dur = LSTM_SEQ_LEN / FPS_TARGET   # seconds per window

    for row_idx, (tid, preds) in enumerate(sorted(by_track.items())):
        for p in preds:
            color = '#e74c3c' if p['is_shoplifting'] else '#2ecc71'
            alpha = min(1.0, 0.4 + p['confidence'] * 0.6)   # opacity = confidence
            ax.barh(row_idx, window_dur, left=p['timestamp'],
                    height=0.6, color=color, alpha=alpha, edgecolor='white')

    ax.set_yticks(range(len(by_track)))
    ax.set_yticklabels([f'Person {tid}' for tid in sorted(by_track.keys())])
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')

    # Legend
    legend_handles = [
        mpatches.Patch(color='#2ecc71', label='Normal'),
        mpatches.Patch(color='#e74c3c', label='Shoplifting'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=10)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Timeline saved to: {save_path}')
    plt.show()
    return fig


def display_results(case: dict):
    """
    Print a formatted summary of the analysis results.

    Designed to be readable by a non-technical security operator.
    All automated scores are accompanied by the human-review disclaimer.
    """
    print('=' * 65)
    print('  DIGITAL WITNESS — ANALYSIS RESULTS')
    print('=' * 65)
    print(f'  Case ID    : {case["case_id"][:8]}...')
    print(f'  Timestamp  : {case["created_at"]}')
    print(f'  Video      : {Path(case["video_metadata"]["path"]).name}')
    print(f'  Duration   : {case["video_metadata"]["duration_seconds"]:.1f}s')
    print(f'  Persons    : {case["video_metadata"]["persons_tracked"]}')
    print('-' * 65)
    print(f'  Classification : {case["overall_result"]["classification"].upper()}')
    print(f'  Confidence     : {case["overall_result"]["confidence"]:.1%}')
    print('-' * 65)
    print('  Intent Score Components:')
    wc = case['intent_score'].get('weighted_components', {})
    for k, v in wc.items():
        print(f'    {k:<14} : {v:.3f}')
    print(f'  FINAL SCORE    : {case["intent_score"]["final_score"]:.3f}')
    print(f'  RISK LEVEL     : {case["intent_score"]["risk_level"]}')
    print('-' * 65)
    print(f'  Alert          : {case["alert"]["alert_level"]}')
    print(f'  Action         : {case["alert"]["action"]}')
    if case.get('pos_comparison'):
        pos = case['pos_comparison']
        print(f'  POS Check      : {pos["flag_type"].upper()} — '
              f'{pos["discrepancy_count"]} item(s) unaccounted for')
    print('-' * 65)
    print(f'  DISCLAIMER: {case["disclaimer"][:80]}...')
    print('=' * 65)

print('Case file functions defined:')
print('  build_case_file(video_path, inference_result, intent_score, alert, pos=None)')
print('  save_case_file(case)')
print('  plot_behavior_timeline(predictions, title, save_path=None)')
print('  display_results(case)')

Case file functions defined:
  build_case_file(video_path, inference_result, intent_score, alert, pos=None)
  save_case_file(case)
  plot_behavior_timeline(predictions, title, save_path=None)
  display_results(case)


## Cell 12 — Full Pipeline with POS Operator Verification

Combines inference (Cell 9), intent scoring (Cell 10), and POS cross-checking (Cell 8) into a single callable function.

**Usage:**
```python
# Basic analysis (video only):
case_id, path, results = analyze_video('path/to/video.mp4')

# With POS integration (matches video timestamp to POS database):
case_id, path, results = analyze_video_with_pos('path/to/video.mp4')
```

In [25]:
# ── CELL 12 — End-to-End Analysis with POS Verification ──────────────────────

def analyze_video(video_path, quality_score: float = 1.0,
                  frame_step: int = 1) -> tuple:
    """
    Orchestrate the full Digital Witness pipeline on a single video.

    Steps:
        1. run_inference()        — YOLO + MobileNetV2 + BiLSTM
        2. calculate_intent_score() — decomposed scoring
        3. bias_aware_adjustment()  — quality-based fairness correction
        4. detect_edge_cases()      — anomaly flags
        5. generate_alert()         — structured alert
        6. build_case_file()        — forensic audit trail
        7. save_case_file()         — persist to outputs/cases/
        8. plot_behavior_timeline() — visualisation
        9. display_results()        — formatted print

    Parameters:
        video_path    : str or Path — input video
        quality_score : float — evidence quality [0, 1] (default: 1.0 = ideal)
        frame_step    : int — process every Nth frame

    Returns:
        tuple (case_id, case_path, full_results_dict)
    """
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f'Video not found: {video_path}')

    print(f'Analysing: {video_path.name}')
    print(f'Quality score: {quality_score}  |  Frame step: {frame_step}')
    print('─' * 50)

    # Step 1: Full inference
    inference_result = run_inference(video_path, frame_step=frame_step)
    print(f'Inference complete: {inference_result["detections"]["frames_processed"]} frames, '
          f'{inference_result["detections"]["persons_tracked"]} persons tracked')

    # Step 2 & 3: Scoring + bias adjustment
    raw_score    = calculate_intent_score(inference_result)
    intent_score = bias_aware_adjustment(raw_score, quality_score=quality_score)

    # Step 4 & 5: Edge cases + alert
    edge_cases = detect_edge_cases(inference_result, intent_score)
    alert      = generate_alert(intent_score, edge_cases)

    # Step 6 & 7: Case file
    case      = build_case_file(video_path, inference_result, intent_score, alert)
    case_path = save_case_file(case)
    print(f'Case file saved: {case_path}')

    # Step 8 & 9: Visualisation + results
    timeline_path = OUTPUTS_DIR / f'timeline_{case["case_id"][:8]}.png'
    plot_behavior_timeline(
        inference_result['predictions'],
        title=f'Behaviour Timeline — {video_path.name}',
        save_path=timeline_path,
    )
    display_results(case)

    return case['case_id'], case_path, {
        'inference':  inference_result,
        'score':      intent_score,
        'alert':      alert,
        'case':       case,
    }


def analyze_video_with_pos(video_path, quality_score: float = 1.0,
                            frame_step: int = 1) -> tuple:
    """
    Full pipeline with POS transaction cross-checking.

    Extends analyze_video() by:
        1. Showing YOLO product detection summary per tracked person
        2. Matching video timestamp to MockPOSDatabase
        3. Calling compare_with_pos() for each tracked person
        4. Adding POS comparison to the case file and intent score

    In a production deployment, MockPOSDatabase would be replaced by
    a real POS API client — the fusion logic is POS-agnostic.

    Parameters:
        video_path    : str or Path — input video
        quality_score : float — evidence quality [0, 1]
        frame_step    : int — process every Nth frame

    Returns:
        tuple (case_id, case_path, full_results_dict)
    """
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f'Video not found: {video_path}')

    print(f'Analysing (with POS): {video_path.name}')
    print('─' * 50)

    # Step 1: Inference
    inference_result = run_inference(video_path, frame_step=frame_step)
    person_sums      = inference_result.get('person_summaries', {})

    # Step 2: Show product detection summary per person
    print('Product Detection Summary per Person:')
    for tid, summary in person_sums.items():
        print(f'  Person {tid}: max_products={summary["max_products_held"]}, '
              f'concealment_frames={summary["concealment_frames"]}, '
              f'checkout_visited={summary["checkout_visited"]}')

    # Step 3: POS matching — use video modification timestamp as proxy
    pos_db    = MockPOSDatabase()
    video_ts  = datetime.datetime.now().isoformat()   # in production: from video metadata
    pos_tx    = pos_db.match_by_timestamp(video_ts)

    # Step 4: POS comparison — use the person with most product interactions
    best_person_sum = None
    if person_sums:
        best_person_sum = max(person_sums.values(),
                              key=lambda s: s['max_products_held'])

    pos_comparison = None
    if best_person_sum:
        pos_comparison = compare_with_pos(best_person_sum, pos_tx)
        print(f'\nPOS Comparison: {pos_comparison["flag_type"].upper()}')
        print(f'  Detected: {pos_comparison["items_detected"]} items')
        print(f'  Billed  : {pos_comparison["items_billed"]} items')
        print(f'  Discrepancy: {pos_comparison["discrepancy_count"]}')

    # Step 5: Scoring with POS data
    raw_score    = calculate_intent_score(inference_result, pos_comparison)
    intent_score = bias_aware_adjustment(raw_score, quality_score=quality_score)

    # Step 6-9: Alert, case file, visualisation
    edge_cases = detect_edge_cases(inference_result, intent_score)
    alert      = generate_alert(intent_score, edge_cases)

    case      = build_case_file(video_path, inference_result,
                                 intent_score, alert, pos_comparison)
    case_path = save_case_file(case)
    print(f'\nCase file saved: {case_path}')

    timeline_path = OUTPUTS_DIR / f'timeline_{case["case_id"][:8]}.png'
    plot_behavior_timeline(
        inference_result['predictions'],
        title=f'Behaviour Timeline — {video_path.name} (with POS)',
        save_path=timeline_path,
    )
    display_results(case)

    return case['case_id'], case_path, {
        'inference':       inference_result,
        'score':           intent_score,
        'alert':           alert,
        'pos_comparison':  pos_comparison,
        'case':            case,
    }


# ── USAGE EXAMPLES ────────────────────────────────────────────────────────────
print('Pipeline ready. Example usage:')
print()
print('  # Basic video analysis:')
print('  case_id, path, results = analyze_video("path/to/video.mp4")')
print()
print('  # With POS cross-checking:')
print('  case_id, path, results = analyze_video_with_pos("path/to/video.mp4")')
print()
print('  # Lower quality score for poor lighting:')
print('  case_id, path, results = analyze_video("path/to/video.mp4", quality_score=0.6)')

Pipeline ready. Example usage:

  # Basic video analysis:
  case_id, path, results = analyze_video("path/to/video.mp4")

  # With POS cross-checking:
  case_id, path, results = analyze_video_with_pos("path/to/video.mp4")

  # Lower quality score for poor lighting:
  case_id, path, results = analyze_video("path/to/video.mp4", quality_score=0.6)


In [26]:
# ── STANDALONE EVALUATION GRAPHS ─────────────────────────────────────────────
# Generates 4 evaluation graphs and saves them to outputs/cases/
# No variables from previous cells needed — runs from saved JSON files only.

import json, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

OUTPUT_DIR = Path("outputs/cases")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load MobileNetV2 metrics from saved JSON
with open("models/mobilenet_dw_info.json") as f:
    mn_info = json.load(f)

cm_data     = mn_info["confusion_matrix"]          # [[2100, 46], [98, 4017]]
acc         = mn_info["metrics"]["accuracy"]        # 0.977
prec        = mn_info["metrics"]["precision"]       # 0.977277
rec         = mn_info["metrics"]["recall"]          # 0.977
f1          = mn_info["metrics"]["f1_score"]        # 0.977065
prec_normal = mn_info["per_class_metrics"]["precision"]["normal"]      # 0.9554
prec_shop   = mn_info["per_class_metrics"]["precision"]["shoplifting"] # 0.9887
rec_normal  = mn_info["per_class_metrics"]["recall"]["normal"]         # 0.9786
rec_shop    = mn_info["per_class_metrics"]["recall"]["shoplifting"]    # 0.9762
f1_normal   = mn_info["per_class_metrics"]["f1"]["normal"]             # 0.9669
f1_shop     = mn_info["per_class_metrics"]["f1"]["shoplifting"]        # 0.9824

# Load BiLSTM metrics from saved JSON
with open("models/bilstm_dw_info.json") as f:
    bl_info = json.load(f)

bl_acc  = bl_info["best_val_acc"]   # 0.8859
bl_seq  = bl_info["seq_len"]        # 45
bl_hid  = bl_info["hidden"]         # 256
bl_lay  = bl_info["layers"]         # 2

# ── GRAPH 1: MobileNetV2 Confusion Matrix ─────────────────────────────────────
cm = np.array(cm_data)
total = cm.sum()
cm_pct = cm / total * 100

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=False, fmt="d", cmap="Blues",
    xticklabels=["Normal", "Shoplifting"],
    yticklabels=["Normal", "Shoplifting"],
    linewidths=0.5, linecolor="white",
    ax=ax
)
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.4, f"{cm[i,j]:,}",
                ha="center", va="center", fontsize=14,
                fontweight="bold",
                color="white" if cm[i,j] > cm.max()/2 else "black")
        ax.text(j + 0.5, i + 0.62, f"({cm_pct[i,j]:.1f}%)",
                ha="center", va="center", fontsize=10,
                color="white" if cm[i,j] > cm.max()/2 else "black")

ax.set_xlabel("Predicted Label", fontsize=12, labelpad=10)
ax.set_ylabel("True Label", fontsize=12, labelpad=10)
ax.set_title(
    f"MobileNetV2 \u2014 Confusion Matrix (Validation Set, n={total:,})\n"
    f"Accuracy: {acc:.1%}  |  Precision: {prec:.1%}  |  "
    f"Recall: {rec:.1%}  |  F1: {f1:.1%}",
    fontsize=11, pad=15
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mobilenet_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: mobilenet_confusion_matrix.png")

# ── GRAPH 2: MobileNetV2 Per-Class Metrics Bar Chart ──────────────────────────
metrics_labels = ["Precision", "Recall", "F1-Score"]
normal_vals    = [prec_normal, rec_normal, f1_normal]
shop_vals      = [prec_shop,   rec_shop,   f1_shop]

x = np.arange(len(metrics_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, normal_vals, width,
               label="Normal", color="#2196F3", alpha=0.85)
bars2 = ax.bar(x + width/2, shop_vals,   width,
               label="Shoplifting", color="#F44336", alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=10)

ax.set_ylim(0.90, 1.01)
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("MobileNetV2 \u2014 Per-Class Performance Metrics\n"
             "Validation Set | Binary Classification: Normal vs Shoplifting",
             fontsize=11, pad=12)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mobilenet_metrics_bar.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: mobilenet_metrics_bar.png")

# ── GRAPH 3: BiLSTM Architecture and Accuracy Summary ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.axis("off")

summary_lines = [
    ("Best Validation Accuracy",   f"{bl_acc:.1%}"),
    ("Sequence Length",            f"{bl_seq} frames  (7.5 seconds @ 6fps)"),
    ("BiLSTM Layers",              f"{bl_lay} stacked layers"),
    ("Hidden Units per Direction", f"{bl_hid}  \u2192  {bl_hid * 2} concatenated"),
    ("Dropout",                    "0.3"),
    ("Attention",                  "Additive temporal attention (Linear \u2192 Softmax)"),
    ("Feature Input",              "1,280-dim MobileNetV2 features per frame"),
    ("Output",                     "Binary: Normal / Shoplifting + attention weights"),
    ("Training Optimiser",         "Adam  |  LR: 5\u00d710\u207b\u2074  |  Weight decay: 1\u00d710\u207b\u2074"),
    ("Train / Val / Test Split",   "80% / 10% / 10%  (stratified by class)"),
]

y_start = 0.92
for label, value in summary_lines:
    ax.text(0.02, y_start, f"{label}:", fontsize=11,
            fontweight="bold", transform=ax.transAxes, va="top")
    ax.text(0.45, y_start, value, fontsize=11,
            transform=ax.transAxes, va="top")
    y_start -= 0.088

ax.set_title("BiLSTM + Temporal Attention \u2014 Model Summary",
             fontsize=13, fontweight="bold", pad=15)
ax.add_patch(mpatches.FancyBboxPatch(
    (0, 0), 1, 1, boxstyle="round,pad=0.02",
    linewidth=1.5, edgecolor="#1A5276", facecolor="#EBF5FB",
    transform=ax.transAxes, zorder=0
))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bilstm_summary.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: bilstm_summary.png")

# ── GRAPH 4: Benchmark Comparison ─────────────────────────────────────────────
systems = [
    "Kim et al. (2021)\nIntent Inference",
    "Archana et al. (2024)\nCNN-LSTM BLRS",
    "Nazir et al. (2023)\nTime-Series F1",
    "Jebur et al. (2025)\nDFF Multi-task",
    "Digital Witness\nMobileNetV2 \u2605",
    "Digital Witness\nBiLSTM \u2605",
]
scores  = [93.1, 95.3, 92.0, 88.4, 97.7, 88.6]
colours = ["#90CAF9","#90CAF9","#90CAF9","#90CAF9","#1565C0","#1565C0"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(systems, scores, color=colours, edgecolor="white", height=0.6)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{score:.1f}%", va="center", fontsize=11, fontweight="bold")

ax.set_xlim(80, 102)
ax.set_xlabel("Accuracy / F1-Score (%)", fontsize=12)
ax.set_title("Benchmark Comparison \u2014 Digital Witness vs Published Systems\n"
             "\u2605 This work  |  All others: published results on UCF-Crime dataset",
             fontsize=11, pad=14)
ax.axvline(x=90, color="grey", linestyle="--", alpha=0.5, linewidth=1)
ax.grid(axis="x", alpha=0.3)

legend_patches = [
    mpatches.Patch(color="#1565C0", label="Digital Witness (This work)"),
    mpatches.Patch(color="#90CAF9", label="Published literature"),
]
ax.legend(handles=legend_patches, fontsize=10, loc="lower right")
ax.text(0.99, 0.02,
        "* Direct comparison is limited \u2014 different datasets,\n"
        "  tasks, and evaluation protocols across studies.",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=8, color="grey", style="italic")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "benchmark_comparison.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: benchmark_comparison.png")



# ── GRAPH 5: MobileNetV2 Training Curves ──────────────────────────────────────
mn_hist_path = Path('models/mobilenet_history.json')
if mn_hist_path.exists():
    with open(mn_hist_path) as hf:
        mn_hist = json.load(hf)

    epochs_mn = range(1, len(mn_hist['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_mn, mn_hist['train_loss'], label='Train Loss', color='#1565C0')
    axes[0].plot(epochs_mn, mn_hist['val_loss'],   label='Val Loss',   color='#EF5350')
    axes[0].set_title('MobileNetV2 -- Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_mn, [a*100 for a in mn_hist['train_acc']], label='Train Acc', color='#1565C0')
    axes[1].plot(epochs_mn, [a*100 for a in mn_hist['val_acc']],   label='Val Acc',   color='#EF5350')
    axes[1].set_title('MobileNetV2 -- Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('MobileNetV2 Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'mobilenet_training_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: mobilenet_training_curves.png')
else:
    print('Skipped MobileNetV2 curves -- models/mobilenet_history.json not found')

# ── GRAPH 6: BiLSTM Training Curves ───────────────────────────────────────────
bl_hist_path = Path('models/bilstm_history.json')
if bl_hist_path.exists():
    with open(bl_hist_path) as hf:
        bl_hist = json.load(hf)

    epochs_bl = range(1, len(bl_hist['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_bl, bl_hist['train_loss'], label='Train Loss', color='#1565C0')
    axes[0].set_title('BiLSTM -- Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_bl, [a*100 for a in bl_hist['train_acc']], label='Train Acc', color='#1565C0')
    axes[1].plot(epochs_bl, [a*100 for a in bl_hist['val_acc']],   label='Val Acc',   color='#EF5350')
    axes[1].set_title('BiLSTM -- Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('BiLSTM Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'bilstm_training_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: bilstm_training_curves.png')
else:
    print('Skipped BiLSTM curves -- models/bilstm_history.json not found')

print("\nAll graphs saved to", OUTPUT_DIR)


Saved: mobilenet_confusion_matrix.png
Saved: mobilenet_metrics_bar.png
Saved: bilstm_summary.png
Saved: benchmark_comparison.png
Saved: mobilenet_training_curves.png
Saved: bilstm_training_curves.png

All graphs saved to outputs\cases
